# 02 · Effectiveness of the DM Remission programme

Analysis of the combined dataset built in `01_combine_data.ipynb`
(`data/processed/dmr_master.csv`).

## What is being asked, and how it is answered

| Question | Approach |
|---|---|
| Who joined the programme? | Descriptive baseline table, overall and by outcome |
| Did glycaemia, weight and waist improve? | Paired within-person change: mean difference with 95 % CI, paired *t* and Wilcoxon signed-rank, standardised effect size |
| Did more patients reach target? | McNemar's test on paired binary targets |
| How many came off their diabetes drugs? | Proportion with Wilson 95 % CI, overall and by site |
| **Who is likely to be drug-stoppable?** | **Multivariable logistic regression, clustered by facility — fitted separately in each cohort** |
| Does the finding replicate outside the programme? | The same model refitted in the รพ.บ้านตาขุน register (§7.9) |
| How do the other cohorts compare? | The registry as a comparison series (§10); the 2566 survey at its own 3-month anchor (§5.3) |
| What happens to cost? | Lab and drug spending per visit, before vs after (§7) |
| What drove the HbA1c improvement? | Linear mixed model with a facility random intercept |

## Two cohorts, analysed in parallel

Both are analysable for drug cessation, and both are reported — but they are
**not pooled**, for a reason that is structural rather than editorial: the
registry is a **single facility**, so a cohort term and a site term cannot be
told apart in any model (§7.9 shows this failing in practice).

* **Programme cohort** — 995 patients, 20 sites, the DM-Remission programme.
* **Registry cohort** — 1,461 patients, รพ.บ้านตาขุน's diabetes register indexed
  at 1 Oct 2023, median follow-up ≈ 28 months. Includes patients later recorded
  as deceased, lost to follow-up or transferred, which is precisely what makes
  it a register rather than a programme roster.

## Design and its limits — read before the numbers

This is a **before-and-after analysis of routine programme data with no
randomised control.** Three consequences follow, and they apply to every
estimate below:

1. **Regression to the mean.** Patients are enrolled partly *because* their
   control is poor, so their next measurement tends to be better regardless of
   any treatment. Some of the HbA1c fall is this artefact.
2. **Secular trend and co-intervention.** Diabetes care changed over the same
   period for reasons unrelated to this programme.
3. **Selection.** Patients offered and accepting deprescribing are the ones
   clinicians judged likely to succeed. The cessation rate is a rate *in those
   selected*, not a rate achievable in all patients with diabetes.

The registry gives a crude comparison series (§8) which partially addresses
(1) and (2) — but it is a different hospital, a different period and a
different measurement schedule, so it is a weak counterfactual, not a control
arm. The regression models identify **associations that predict who succeeds**;
they do not estimate a causal effect of the programme.

## 0 · Setup

In [ ]:
import sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats

warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path.cwd().parent))
from src import dmr_viz

dmr_viz.use_theme()
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 80)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

ROOT = Path.cwd().parent
FIG = ROOT / "outputs" / "figures"
TAB = ROOT / "outputs" / "tables"
FIG.mkdir(parents=True, exist_ok=True)
TAB.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(ROOT / "data" / "processed" / "dmr_master.csv",
                 parse_dates=["dx_date", "enroll_date"])
print(f"{len(df):,} patients × {df.shape[1]} columns")
print(df["cohort"].value_counts().to_string())

### 0.1 Statistical helpers

Written out here rather than hidden in a module, because the choice of
interval and test is part of what is being reviewed.

* **Wilson interval** for proportions — behaves properly at rates near 0 and 1,
  where the normal approximation gives intervals crossing the boundary. The
  per-site cessation rates include a 0 % and a 100 %, so this matters.
* **Paired change** is reported with *both* a paired *t*-test and a Wilcoxon
  signed-rank test. HbA1c and FBS are right-skewed; when the two tests agree,
  the conclusion does not rest on the normality assumption.
* **Cohen's *d*_z** is the standardised paired effect — the mean change divided
  by the SD *of the change*, which is the right denominator for paired data.

In [ ]:
def wilson_ci(k, n, conf=0.95):
    '''Wilson score interval for a binomial proportion.'''
    if n == 0:
        return (np.nan, np.nan, np.nan)
    z = stats.norm.ppf(1 - (1 - conf) / 2)
    p = k / n
    den = 1 + z**2 / n
    centre = (p + z**2 / (2 * n)) / den
    half = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / den
    return p, max(0.0, centre - half), min(1.0, centre + half)


def prop_row(label, series):
    '''Proportion of a 0/1 series with its Wilson interval.'''
    s = pd.to_numeric(series, errors="coerce").dropna()
    p, lo, hi = wilson_ci(int(s.sum()), len(s))
    return {"outcome": label, "n": len(s), "events": int(s.sum()),
            "pct": 100 * p, "lo95": 100 * lo, "hi95": 100 * hi}


def paired_change(data, pre, post, label, unit=""):
    '''
    Within-person change on complete pairs.

    Returns the paired mean difference with its 95 % CI, the paired t-test and
    the Wilcoxon signed-rank test, and Cohen's d_z.
    '''
    d = data[[pre, post]].dropna()
    if len(d) < 3:
        return None
    a, b = d[pre].to_numpy(float), d[post].to_numpy(float)
    diff = b - a
    n = len(diff)
    sd = diff.std(ddof=1)
    se = sd / np.sqrt(n)
    tcrit = stats.t.ppf(0.975, n - 1)
    t_p = stats.ttest_rel(b, a).pvalue
    w_p = (stats.wilcoxon(b, a).pvalue if np.any(diff != 0) else np.nan)
    return {
        "measure": label, "unit": unit, "n_pairs": n,
        "pre_mean": a.mean(), "pre_sd": a.std(ddof=1),
        "post_mean": b.mean(), "post_sd": b.std(ddof=1),
        "mean_change": diff.mean(),
        "lo95": diff.mean() - tcrit * se, "hi95": diff.mean() + tcrit * se,
        "cohen_dz": diff.mean() / sd if sd > 0 else np.nan,
        "p_ttest": t_p, "p_wilcoxon": w_p,
    }


def mcnemar_paired(data, pre, post, label):
    '''McNemar's test on a paired binary target (exact when discordants < 25).'''
    d = data[[pre, post]].dropna()
    if len(d) < 3:
        return None
    a, b = d[pre].astype(int), d[post].astype(int)
    b01 = int(((a == 0) & (b == 1)).sum())   # gained the target
    b10 = int(((a == 1) & (b == 0)).sum())   # lost it
    nd = b01 + b10
    if nd == 0:
        p = 1.0
    elif nd < 25:
        p = float(stats.binomtest(b01, nd, 0.5).pvalue)
    else:
        p = float(stats.chi2.sf((abs(b01 - b10) - 1) ** 2 / nd, 1))
    return {"target": label, "n_pairs": len(d),
            "pct_pre": 100 * a.mean(), "pct_post": 100 * b.mean(),
            "gained": b01, "lost": b10, "p_mcnemar": p}


def pct_ci(series):
    """A 0/1 series as 'pct% (95% CI lo-hi; n=…)', or a clear note when empty."""
    s = pd.to_numeric(series, errors="coerce").dropna()
    if len(s) == 0:
        return "not measurable"
    p, lo, hi = wilson_ci(int(s.sum()), len(s))
    return f"{100*p:.1f}% (95% CI {100*lo:.1f}-{100*hi:.1f}; n={len(s)})"


def fmt_p(p):
    if pd.isna(p):
        return "—"
    return "<0.001" if p < 0.001 else f"{p:.3f}"


print("helpers defined")

---
## 1 · Analysis sets

Every result below states which population it is on.

* **Programme cohort** (`cohort == "program"`) — 995 patients, 20 sites. All
  headline effectiveness results.
* **Registry cohort** (`cohort == "registry"`) — 1,461 patients, one hospital.
  Reported alongside, never pooled.
* **Drug-cessation analysis set** — within each cohort, patients who were *on a
  glucose-lowering drug at baseline* and whose follow-up regimen is known.
  Patients already drug-free at enrolment cannot "stop", so including them
  would pad the denominator with people never at risk of the outcome.

In [ ]:
prog = df[df["cohort"] == "program"].copy()
reg  = df[df["cohort"] == "registry"].copy()
elig = prog[prog["drug_stop_eligible"] == True].copy()
elig_reg = reg[reg["drug_stop_eligible"] == True].copy()

def flow_for(data, name):
    return pd.DataFrame([
        (f"{name}: all records", len(data)),
        ("  - not on a glucose-lowering drug at baseline",
         -int((data["on_any_pre"] == 0).sum())),
        ("  - baseline regimen not recorded", -int(data["on_any_pre"].isna().sum())),
        ("  - follow-up regimen not recoverable",
         -int(((data["on_any_pre"] == 1) & data["on_any_last"].isna()).sum())),
        ("  = drug-cessation analysis set",
         int(data["drug_stop_eligible"].sum())),
    ], columns=["step", "n"])

flow = pd.concat([flow_for(prog, "Programme cohort"),
                  flow_for(reg, "Registry cohort")], ignore_index=True)
flow

In [ ]:
for nm, d_, e_ in [("Programme", prog, elig), ("Registry ", reg, elig_reg)]:
    print(f"{nm}: {len(d_):,} patients in {d_['facility'].nunique()} site(s); "
          f"cessation set {len(e_):,}, {int(e_['drug_stopped'].sum())} events "
          f"({100*e_['drug_stopped'].mean():.1f}%)")

print("\nRegistry membership status (why it is a register, not a roster):")
print(reg["member_status"].value_counts(dropna=False).to_string())
print(f"\nRegistry follow-up, months: median "
      f"{reg['followup_months'].median():.1f} "
      f"(IQR {reg['followup_months'].quantile(.25):.1f}-"
      f"{reg['followup_months'].quantile(.75):.1f})")
print("\nProgramme cohort sites:")
print(prog["facility"].value_counts().rename("patients").to_frame().to_string())

---
## 2 · Descriptive results

### 2.1 Table 1 — baseline characteristics

Continuous variables are summarised as mean (SD), categorical as n (%).
Columns split the programme cohort by the primary outcome; the *p* column is an
**unadjusted** comparison (Welch's *t* for continuous, χ² for categorical),
shown for orientation only — the adjusted, clustered comparison is §7.

Denominators differ between rows because missingness is structural: the
รพ.สต. template records no diagnosis date, benefit scheme or comorbidity at
all. Each row's *n* is printed so this is visible rather than hidden.

In [ ]:
def table1(data, group=None, cont=(), cat=()):
    '''Descriptive table; optional two-level grouping with an unadjusted test.'''
    rows = []
    if group is not None:
        g = data[group]
        levels = [l for l in sorted(g.dropna().unique())]
        parts = [data[g == l] for l in levels]
        names = [f"{group}={l:g}" if isinstance(l, float) else f"{group}={l}"
                 for l in levels]
    else:
        parts, names = [], []

    rows.append({"variable": "N", "n": len(data), "overall": f"{len(data)}",
                 **{nm: f"{len(p)}" for nm, p in zip(names, parts)}, "p": ""})

    for v, lab in cont:
        s = pd.to_numeric(data[v], errors="coerce")
        r = {"variable": f"{lab}, mean (SD)", "n": int(s.notna().sum()),
             "overall": f"{s.mean():.1f} ({s.std():.1f})"}
        vals = []
        for nm, p in zip(names, parts):
            x = pd.to_numeric(p[v], errors="coerce").dropna()
            vals.append(x)
            r[nm] = f"{x.mean():.1f} ({x.std():.1f})" if len(x) else "-"
        if len(vals) == 2 and min(len(v_) for v_ in vals) >= 3:
            r["p"] = fmt_p(stats.ttest_ind(*vals, equal_var=False).pvalue)
        else:
            r["p"] = ""
        rows.append(r)

    for v, lab in cat:
        s = data[v].dropna()
        rows.append({"variable": f"{lab}, n (%)", "n": int(len(s)),
                     "overall": "", **{nm: "" for nm in names}, "p": ""})
        cats = list(pd.Series(s.unique()).sort_values())
        tab = []
        for c in cats:
            r = {"variable": f"    {c}", "n": int((s == c).sum()),
                 "overall": f"{(s == c).sum()} ({100*(s == c).mean():.1f}%)"}
            col = []
            for nm, p in zip(names, parts):
                ps = p[v].dropna()
                k = int((ps == c).sum())
                col.append(k)
                r[nm] = f"{k} ({100*k/len(ps):.1f}%)" if len(ps) else "-"
            tab.append(col)
            r["p"] = ""
            rows.append(r)
        if len(names) == 2 and len(cats) > 1:
            arr = np.array(tab)
            if arr.min() >= 0 and arr.sum() > 0:
                try:
                    pval = stats.chi2_contingency(arr[arr.sum(axis=1) > 0])[1]
                    rows[-len(cats) - 1]["p"] = fmt_p(pval)
                except ValueError:
                    pass
    return pd.DataFrame(rows).fillna("")


CONT = [("age", "Age, y"), ("dm_duration_y", "Diabetes duration, y"),
        ("bmi_pre", "BMI, kg/m²"), ("waist_pre", "Waist circumference, cm"),
        ("fbs_pre", "Fasting blood sugar, mg/dL"), ("hba1c_pre", "HbA1c, %"),
        ("n_class_pre", "Glucose-lowering drug classes, n"),
        ("tabs_pre", "Oral tablets per day, n")]
CAT = [("sex", "Sex"), ("age_group", "Age group"), ("scheme", "Benefit scheme"),
       ("bmi_cat", "BMI category"), ("a1c_cat_pre", "Baseline HbA1c band"),
       ("dm_duration_cat", "Diabetes duration band"),
       ("on_insulin_pre", "On insulin at baseline"),
       ("hypertension", "Hypertension"), ("dyslipidaemia", "Dyslipidaemia")]

t1 = table1(prog, cont=CONT, cat=CAT)
t1.to_csv(TAB / "table1_baseline_overall.csv", index=False)
t1

In [ ]:
t1b = table1(elig, group="drug_stopped", cont=CONT, cat=CAT)
t1b.columns = [c.replace("drug_stopped=0", "Still on drugs")
                .replace("drug_stopped=1", "Stopped all drugs") for c in t1b.columns]
t1b.to_csv(TAB / "table1_baseline_by_outcome.csv", index=False)
t1b

### 2.3 Baseline by provider level

The programme ran at two kinds of facility, and they are analysed as subgroups
throughout:

| Level | Meaning | Sites | Patients |
|---|---|---|---|
| `community_hospital` | โรงพยาบาลชุมชน — district hospital with beds, own pharmacy and lab | 5 | 613 |
| `primary_care` | โรงพยาบาลส่งเสริมสุขภาพตำบล (รพ.สต.) — sub-district health centre | 15 | 382 |

Two assignments are not what their name suggests, and both were confirmed with
the programme team:

* **PCC คลองศาลา** is a Primary Care Cluster physically based *inside* a
  community hospital, so it counts as `community_hospital`.
* The workbook labelled **โรงพยาบาลกงหรา** is actually from
  **รพ.สต.บ้านโล๊ะจังกะ**, a health centre in that hospital's network. It is
  renamed and counted as `primary_care`.

The mapping lives in `FACILITY_INFO` in `dmr_etl.py`, which is also where the
raw workbook strings are normalised to canonical facility names.

In [ ]:
t1c = table1(prog, group="provider_level", cont=CONT, cat=CAT)
t1c.columns = [c.replace("provider_level=community_hospital", "Community hospital")
                .replace("provider_level=primary_care", "Primary care") for c in t1c.columns]
t1c.to_csv(TAB / "table1c_baseline_by_provider_level.csv", index=False)
t1c

**Reading Table 1.** Two baseline differences stand out and both are large:
patients who came off their drugs started with **lower HbA1c** and on **fewer
drug classes**. Neither is surprising — it is close to a description of who
clinicians select for deprescribing — and it is exactly why the adjusted model
in §6 matters more than these unadjusted columns.

### 2.2 Baseline distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
specs = [("hba1c_pre", "HbA1c at baseline (%)", 6.5, "ADA target 6.5%"),
         ("fbs_pre", "Fasting blood sugar at baseline (mg/dL)", 126, "126 mg/dL"),
         ("bmi_pre", "BMI at baseline (kg/m²)", 25, "25 kg/m²")]
for ax, (col, title, ref, reflab) in zip(axes, specs):
    s = pd.to_numeric(prog[col], errors="coerce").dropna()
    ax.hist(s, bins=30, color=dmr_viz.BLUE, alpha=0.9)
    ax.axvline(ref, color=dmr_viz.INK_2, lw=1.4, ls="--")
    ax.annotate(reflab, xy=(ref, ax.get_ylim()[1] * 0.92), xytext=(5, 0),
                textcoords="offset points", fontsize=8, color=dmr_viz.INK_2)
    ax.set_title(title, fontsize=10.5)
    ax.set_ylabel("patients")
    ax.annotate(f"n = {len(s)}   median {s.median():.1f}",
                xy=(0.98, 0.92), xycoords="axes fraction", ha="right",
                fontsize=8.5, color=dmr_viz.MUTED)
fig.suptitle("Programme cohort at enrolment", x=0.005, ha="left",
             fontsize=13, fontweight="semibold")
fig.tight_layout(rect=[0, 0, 1, 0.94])
fig.savefig(FIG / "fig01_baseline_distributions.png", bbox_inches="tight")
plt.show()

---
## 3 · The 6-month anchor

**Outcomes are defined at a fixed timepoint — 6 months after enrolment —
rather than at whatever value a site happened to record last.** "Last
observed" conflates a patient measured at 3 months with one measured at 12,
and makes the programme and registry cohorts incomparable because their
follow-up lengths differ by design.

The workbooks support that anchor very unevenly, so **every 6-month value
carries a tier recording how its timing was established**. The assumption is a
column in the data, not a footnote.

| Tier | Meaning | Source |
|---|---|---|
| `measured` | A real measurement date, accepted when it falls within **±3 months** of the 6-month target | รพ.บ้านตาขุน — the only workbook with dates |
| `labelled` | A column the template explicitly labels as *N months since enrolment* | CRF `FBS ระหว่างเข้าร่วมโครงการ 6 เดือน`, บางระกำ `FBS 6 เดือน`, PCC `6 mo.` |
| `assumed` | A single unlabelled follow-up column, taken to be ≈ 6 months | รพ.สต. `หลัง` block (its *risk-group* sheet in the same workbook says `หลังเข้าโครงการ 6 เดือน`); CRF mid-programme columns |

**One trap worth naming.** The CRF's `หลังจบโครงการ 6 เดือน` columns are
anchored to the programme's **end**, not to enrolment — a different clock.
They are *not* used here; the mid-programme column is used instead, at the
`assumed` tier. Similarly, รพ.พิมาย's 22-column CRF has only one unlabelled
mid-programme FBS column, so its 100 patients are `assumed` while the
24-column files' are `labelled`.

In [ ]:
M6 = {"hba1c": "hba1c_m6", "fbs": "fbs_m6", "bmi": "bmi_m6",
      "waist": "waist_m6", "medication": "on_any_m6"}

cov = []
for lab, col in M6.items():
    for coh in ["program", "registry"]:
        sub = df[df["cohort"] == coh]
        for tier in ["measured", "labelled", "assumed"]:
            n = int((sub[f"{col}_tier"] == tier).sum())
            if n:
                cov.append({"measure": lab, "cohort": coh, "tier": tier, "n": n})
cover = (pd.DataFrame(cov).pivot_table(index=["measure", "cohort"], columns="tier",
                                       values="n", fill_value=0)
         .reindex(columns=["measured", "labelled", "assumed"], fill_value=0))
cover["total"] = cover.sum(axis=1)
cover["primary (meas.+lab.)"] = cover["measured"] + cover["labelled"]
cover.to_csv(TAB / "table15_timing_coverage.csv")
cover

**Read this table before any 6-month result.** It sets what each outcome can
honestly support:

* **FBS and the anthropometrics** have a solid `measured`+`labelled` base in
  both cohorts.
* **HbA1c** is `measured` in the registry but `assumed` for almost the whole
  programme cohort — the CRF simply never labels its HbA1c follow-up against
  enrolment.
* **Medication** has no `measured` tier at all: the registry's drug sheet
  carries no dates, so **drug cessation at 6 months cannot be computed for the
  registry**. It exists only in the programme cohort, and mostly at the
  `assumed` tier.

Because the tier differs *by measure*, a per-patient "quality grade" would be
misleading — a CRF patient has `labelled` FBS and `assumed` HbA1c at once. So
every result below reports the tier mix of its own denominator, and restricts
to `measured`+`labelled` wherever that leaves enough data to be worth doing.

In [ ]:
def tier_mix(data, col):
    '''Compact description of where a column's timing came from.'''
    t = data.loc[data[col].notna(), f"{col}_tier"].value_counts()
    return ", ".join(f"{k} {v}" for k, v in t.items()) or "-"


def m6_set(data, col, primary_only=False):
    '''Rows with a 6-month value; optionally only the measured/labelled tiers.'''
    d = data[data[col].notna()]
    if primary_only:
        d = d[d[f"{col}_tier"].isin(["measured", "labelled"])]
    return d


for coh, data in [("Programme", prog), ("Registry", reg)]:
    print(f"--- {coh} ---")
    for lab, col in M6.items():
        n_all = int(data[col].notna().sum())
        n_pri = len(m6_set(data, col, True))
        print(f"  {lab:11s} any tier {n_all:5d} | measured+labelled {n_pri:5d}"
              f"   [{tier_mix(data, col)}]")

---
## 4 · Programme effectiveness — within-person change

Complete pairs only: a patient contributes a row only if both the baseline and
the follow-up value are present.

Each measure is reported **twice**, so the effect of the definition is visible
rather than buried:

* **At 6 months** — the time-bound definition (§3). This is the headline.
* **Last observed** — the previous definition, whatever value the site recorded
  last. Retained as a secondary column because it uses more data.

If the two columns disagree materially, the "last observed" number was being
driven by follow-up length rather than by treatment.

In [ ]:
pairs = [("hba1c_pre", "hba1c_last", "HbA1c", "%"),
         ("fbs_pre", "fbs_last", "Fasting blood sugar", "mg/dL"),
         ("bmi_pre", "bmi_post", "BMI", "kg/m²"),
         ("waist_pre", "waist_post", "Waist circumference", "cm"),
         ("weight_pre", "weight_post", "Body weight", "kg"),
         ("tabs_pre", "tabs_post", "Oral tablets per day", "n")]

# the same measures on the 6-month anchor
pairs_m6 = [("hba1c_pre", "hba1c_m6", "HbA1c", "%"),
            ("fbs_pre", "fbs_m6", "Fasting blood sugar", "mg/dL"),
            ("bmi_pre", "bmi_m6", "BMI", "kg/m²"),
            ("waist_pre", "waist_m6", "Waist circumference", "cm")]


def change_table(data, spec, label):
    rows = []
    for a, b, lab, u in spec:
        r = paired_change(data, a, b, lab, u)
        if r:
            r["definition"] = label
            r["tiers"] = tier_mix(data.dropna(subset=[a, b]), b) if b.endswith("_m6") else "-"
            rows.append(r)
    return pd.DataFrame(rows)


chg_m6 = change_table(prog, pairs_m6, "At 6 months")
chg = change_table(prog, pairs, "Last observed")
both = pd.concat([chg_m6, chg], ignore_index=True)
both.to_csv(TAB / "table2_paired_change.csv", index=False)

disp = both.assign(
    change=lambda d: d.mean_change.round(2).astype(str) + " ("
                     + d.lo95.round(2).astype(str) + " to " + d.hi95.round(2).astype(str) + ")",
    p=lambda d: d.p_wilcoxon.map(fmt_p))
disp = disp.pivot_table(index="measure", columns="definition",
                        values=["n_pairs", "mean_change"], aggfunc="first")
disp.columns = [f"{b} · {a}" for a, b in disp.columns]
disp.round(2)

Full detail for the time-bound definition, including the timing tiers behind
each row:

In [ ]:
chg_m6.assign(
    pre=lambda d: d.pre_mean.round(2).astype(str) + " (" + d.pre_sd.round(2).astype(str) + ")",
    post=lambda d: d.post_mean.round(2).astype(str) + " (" + d.post_sd.round(2).astype(str) + ")",
    change=lambda d: d.mean_change.round(2).astype(str) + " ("
                     + d.lo95.round(2).astype(str) + " to " + d.hi95.round(2).astype(str) + ")",
    dz=lambda d: d.cohen_dz.round(2),
    p_t=lambda d: d.p_ttest.map(fmt_p), p_w=lambda d: d.p_wilcoxon.map(fmt_p),
)[["measure", "unit", "n_pairs", "pre", "post", "change", "dz", "p_t", "p_w", "tiers"]].rename(
    columns={"measure": "Measure", "unit": "Unit", "n_pairs": "n pairs",
             "pre": "Baseline mean (SD)", "post": "6-month mean (SD)",
             "change": "Mean change (95% CI)", "dz": "Cohen's dz",
             "p_t": "p (t)", "p_w": "p (Wilcoxon)", "tiers": "Timing tiers"})

#### Restricted to `measured` + `labelled` timing

The same change, dropping every patient whose 6-month timing is an assumption.
For HbA1c in the programme cohort this leaves very little, which is itself the
result: **the programme's HbA1c timing is essentially never recorded.**

In [ ]:
rows = []
for a, b, lab, u in pairs_m6:
    sub = m6_set(prog, b, primary_only=True)
    r = paired_change(sub, a, b, lab, u)
    if r:
        r["tiers"] = tier_mix(sub.dropna(subset=[a, b]), b)
        rows.append(r)
strict_m6 = pd.DataFrame(rows)
strict_m6.to_csv(TAB / "table2b_paired_change_m6_strict.csv", index=False)
(strict_m6.assign(change=lambda d: d.mean_change.round(2).astype(str) + " ("
                  + d.lo95.round(2).astype(str) + " to " + d.hi95.round(2).astype(str) + ")",
                  p=lambda d: d.p_wilcoxon.map(fmt_p))
 [["measure", "unit", "n_pairs", "change", "p", "tiers"]]
 .rename(columns={"measure": "Measure", "unit": "Unit", "n_pairs": "n pairs",
                  "change": "Mean change (95% CI)", "p": "p (Wilcoxon)",
                  "tiers": "Timing tiers"}))

In [ ]:
fig, ax = plt.subplots(figsize=(8.8, 3.6))
sub = chg_m6.copy()
y = np.arange(len(sub))[::-1]
colors = [dmr_viz.BLUE if v < 0 else dmr_viz.DIV_BAD for v in sub["cohen_dz"]]
ax.barh(y, sub["cohen_dz"], color=colors, height=0.55)
ax.axvline(0, color=dmr_viz.AXIS, lw=1)
ax.set_yticks(y)
ax.set_yticklabels(sub["measure"])
ax.set_xlabel("Cohen's $d_z$  (standardised paired change; negative = improvement)")
for yi, (dz, n, mc, u, lo, hi) in enumerate(
        zip(sub["cohen_dz"], sub["n_pairs"], sub["mean_change"],
            sub["unit"], sub["lo95"], sub["hi95"])):
    off = -6 if dz < 0 else 6
    ha = "right" if dz < 0 else "left"
    ax.annotate(f"{mc:+.2f} {u}  (95% CI {lo:+.2f} to {hi:+.2f}; n={n})",
                xy=(dz, y[yi]), xytext=(off, 0), textcoords="offset points",
                va="center", ha=ha, fontsize=8.5, color=dmr_viz.INK_2)
span = max(abs(sub["cohen_dz"].min()), abs(sub["cohen_dz"].max()))
ax.set_xlim(-span * 2.6, span * 2.6)
dmr_viz.bare_y(ax)
ax.set_title("Within-person change at 6 months after enrolment\nProgramme cohort, complete pairs")
dmr_viz.caption(ax, "Timing tiers vary by measure - see §3. HbA1c is `assumed` "
                    "for almost the whole programme cohort.")
fig.tight_layout()
fig.savefig(FIG / "fig02_paired_change.png", bbox_inches="tight")
plt.show()

### 4.1 Distribution of individual change, not just the mean

A mean change hides how many patients moved the wrong way. The chart below
shows every patient's HbA1c change; the shaded region is deterioration.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# (a) histogram of individual HbA1c change at 6 months
s = pd.to_numeric(prog["d_hba1c_m6"], errors="coerce").dropna()
ax = axes[0]
bins = np.arange(np.floor(s.min()), np.ceil(s.max()) + 0.25, 0.25)
n_, b_, patches = ax.hist(s, bins=bins)
for p, left in zip(patches, b_[:-1]):
    p.set_facecolor(dmr_viz.BLUE if left < 0 else dmr_viz.DIV_BAD)
ax.axvline(0, color=dmr_viz.INK_2, lw=1.2)
ax.set_xlabel("Change in HbA1c at 6 months (percentage points)")
ax.set_ylabel("patients")
ax.set_title("(a) Individual HbA1c change at 6 months")
ax.annotate(f"improved  {100*(s<0).mean():.0f}%", xy=(0.03, 0.92),
            xycoords="axes fraction", fontsize=9, color=dmr_viz.BLUE, fontweight="semibold")
ax.annotate(f"worsened  {100*(s>0).mean():.0f}%", xy=(0.03, 0.84),
            xycoords="axes fraction", fontsize=9, color=dmr_viz.DIV_BAD, fontweight="semibold")
ax.annotate(f"n = {len(s)}", xy=(0.97, 0.92), xycoords="axes fraction",
            ha="right", fontsize=8.5, color=dmr_viz.MUTED)

# (b) baseline vs 6 months, showing regression to the mean
ax = axes[1]
d = prog[["hba1c_pre", "hba1c_m6"]].dropna()
ax.scatter(d["hba1c_pre"], d["hba1c_m6"], s=14, alpha=0.45,
           color=dmr_viz.BLUE, edgecolors="none")
lim = [3.5, max(d.max()) + 0.5]
ax.plot(lim, lim, color=dmr_viz.INK_2, lw=1.2, ls="--")
ax.annotate("no change", xy=(lim[1], lim[1]), xytext=(-4, 6),
            textcoords="offset points", ha="right", fontsize=8.5, color=dmr_viz.INK_2)
ax.axhline(6.5, color=dmr_viz.MUTED, lw=1, ls=":")
ax.annotate("6.5%", xy=(lim[0], 6.5), xytext=(2, 3), textcoords="offset points",
            fontsize=8, color=dmr_viz.MUTED)
ax.set_xlim(lim); ax.set_ylim(lim)
ax.set_xlabel("HbA1c at baseline (%)")
ax.set_ylabel("HbA1c at 6 months (%)")
ax.set_title("(b) Baseline vs 6-month HbA1c")
ax.annotate("points below the dashed line improved;\nthe fan shape is the "
            "regression-to-the-mean\nsignature this design cannot remove",
            xy=(0.04, 0.80), xycoords="axes fraction", fontsize=8,
            color=dmr_viz.MUTED, va="top")

fig.tight_layout()
fig.savefig(FIG / "fig03_hba1c_change_distribution.png", bbox_inches="tight")
plt.show()

### 4.2 Proportions reaching target — McNemar's test

Paired binary comparison on the same patients before and after. `gained` is the
number who were off target at baseline and on target at follow-up; `lost` is
the reverse. McNemar's test uses only those discordant pairs, which is the
correct test for paired proportions.

In [ ]:
mc = pd.DataFrame([r for r in [
    mcnemar_paired(prog, "a1c_lt7_pre", "a1c_lt7_m6", "HbA1c < 7% (at 6 months)"),
    mcnemar_paired(prog, "fbs_target_pre", "fbs_target_m6", "FBS < 126 mg/dL (at 6 months)"),
    mcnemar_paired(prog, "a1c_lt7_pre", "a1c_lt7_post", "HbA1c < 7% (last observed)"),
    mcnemar_paired(prog, "fbs_target_pre", "fbs_target_post", "FBS < 126 mg/dL (last observed)"),
] if r is not None])
mc_disp = mc.assign(
    pct_pre=lambda d: d.pct_pre.round(1), pct_post=lambda d: d.pct_post.round(1),
    p=lambda d: d.p_mcnemar.map(fmt_p))[
    ["target", "n_pairs", "pct_pre", "pct_post", "gained", "lost", "p"]]
mc_disp.columns = ["Target", "n pairs", "% at baseline", "% at follow-up",
                   "Gained target", "Lost target", "p (McNemar)"]
mc.to_csv(TAB / "table3_mcnemar_targets.csv", index=False)
mc_disp

| Outcome | Definition | Denominator |
|---|---|---|
| **Drug cessation at 6 months** (primary) | On ≥ 1 glucose-lowering drug at baseline, on none at the 6-month timepoint | Patients with a 6-month medication value |
| Drug cessation, last observed | As above but at whatever regimen was recorded last | Cessation analysis set |
| Dose reduction | Fewer oral tablets per day at follow-up | Those with tablet counts at both points |
| **Remission at 6 months** | HbA1c < 6.5 % at 6 months **and** off all glucose-lowering drugs | Those with both values |

**Drug cessation at 6 months does not exist for the registry.** Its medication
sheet is an undated dispensing list, so there is no way to say what a patient
was taking at month 6 specifically. The registry keeps its last-observed
cessation figure (median follow-up 28 months) and the two are **not** compared
as if they were the same quantity.

**The remission figure is an upper bound.** The ADA/EASD 2021 consensus
requires HbA1c measured at least 3 months *after* drugs stop. The date drugs
stopped is not recorded, so that condition cannot be checked.

In [ ]:
def outcome_block(data, label):
    e6 = data[data["drug_stop_m6_eligible"] == True]
    e = data[data["drug_stop_eligible"] == True]
    rows = [
        prop_row("Drug cessation AT 6 MONTHS (primary)", e6["drug_stopped_m6"]),
        prop_row("Remission AT 6 MONTHS - upper bound", data["remission_m6"]),
        prop_row("HbA1c < 7% at 6 months", data["a1c_lt7_m6"]),
        prop_row("HbA1c < 6.5% at 6 months", data["a1c_lt65_m6"]),
        prop_row("FBS < 126 mg/dL at 6 months", data["fbs_target_m6"]),
        prop_row("- Drug cessation, last observed", e["drug_stopped"]),
        prop_row("- Drug cessation, strict/cost-confirmed", e["drug_stopped_strict"]),
        prop_row("- Dose reduction, last observed", e["dose_reduced"]),
        prop_row("- Remission, last observed", data["remission"]),
        prop_row("- HbA1c < 7%, last observed", data["a1c_lt7_post"]),
    ]
    t = pd.DataFrame(rows)
    t.insert(0, "cohort", label)
    return t

out = pd.concat([outcome_block(prog, "Programme"),
                 outcome_block(reg, "Registry")], ignore_index=True)
out_disp = out.assign(**{"% (95% CI)": lambda d: np.where(
    d.n > 0,
    d.pct.round(1).astype(str) + " (" + d.lo95.round(1).astype(str)
    + "-" + d.hi95.round(1).astype(str) + ")", "not measurable")})[
    ["cohort", "outcome", "events", "n", "% (95% CI)"]]
out_disp.columns = ["Cohort", "Outcome", "Events", "Denominator", "% (95% CI)"]
out.to_csv(TAB / "table4_outcome_rates.csv", index=False)
out_disp

### 5.1 Between-site variation

Sites are not interchangeable, and the spread below is the single most
important descriptive finding in this section: it ranges from 0 % to 100 %.

Some of that is small denominators — the Wilson intervals make that visible —
but the exchangeable correlation estimated in §6 confirms real clustering.
This is why every model that follows accounts for facility, and why a single
pooled rate should not be quoted without this chart beside it.

In [ ]:
elig_m6 = prog[prog["drug_stop_m6_eligible"] == True].copy()

by_site = (elig_m6.groupby("facility")["drug_stopped_m6"]
           .agg(n="count", k="sum").astype(int))
by_site = by_site[by_site["n"] >= 8].copy()
by_site[["p", "lo", "hi"]] = by_site.apply(
    lambda r: pd.Series(wilson_ci(r["k"], r["n"])), axis=1)
by_site = by_site.sort_values("p")

fig, ax = plt.subplots(figsize=(8.6, max(3.4, 0.34 * len(by_site) + 1.8)))
y = np.arange(len(by_site))
ax.hlines(y, 100 * by_site["lo"], 100 * by_site["hi"], color=dmr_viz.AXIS, lw=2.2)
ax.plot(100 * by_site["p"], y, "o", color=dmr_viz.BLUE, markersize=8,
        markeredgecolor=dmr_viz.SURFACE, markeredgewidth=2, linestyle="none")

pooled_p, pooled_lo, pooled_hi = wilson_ci(
    int(elig_m6["drug_stopped_m6"].sum()), int(elig_m6["drug_stopped_m6"].notna().sum()))
ax.axvline(100 * pooled_p, color=dmr_viz.ORANGE, lw=1.6, ls="--", zorder=0)
ax.annotate(f"pooled {100*pooled_p:.1f}%", xy=(100 * pooled_p, len(by_site) - 0.3),
            xytext=(6, 0), textcoords="offset points", fontsize=9,
            color=dmr_viz.ORANGE, fontweight="semibold")

ax.set_yticks(y)
ax.set_yticklabels([f"{f}  ({k}/{n})" for f, n, k in
                    zip(by_site.index, by_site["n"], by_site["k"])], fontsize=8.5)
ax.set_xlabel("Patients off all glucose-lowering drugs at 6 months (%)")
ax.set_xlim(-3, 105)
ax.grid(axis="y", visible=False)
ax.set_title("Drug cessation at 6 months by site, with Wilson 95% intervals\n"
             "Programme cohort, sites with ≥ 8 eligible patients")
fig.tight_layout()
fig.savefig(FIG / "fig04_cessation_by_site.png", bbox_inches="tight")
plt.show()

by_site.round(3).to_csv(TAB / "table5_cessation_by_site.csv")
by_site.assign(pct=(100 * by_site["p"]).round(1))[["n", "k", "pct"]]

### 5.2 By provider level — and why the difference cannot be attributed to it

Cessation at 6 months is roughly **twice as high in primary care** as in
community hospitals. That is a large, statistically clear difference, and it is
the kind of finding a programme evaluation would want to lead with.

**It should not be reported as a provider-level effect** — but the reason is
confounding with the data template, not a clean within-template test that
refutes it. The next cells show the confounding, then show that the test which
would settle it is too small to do so.

In [ ]:
lvl_rows = []
for lv, s in elig_m6.groupby("provider_level"):
    r = prop_row(lv, s["drug_stopped_m6"])
    r["sites"] = s["facility"].nunique()
    lvl_rows.append(r)
lvl = pd.DataFrame(lvl_rows)

tab = [[int(s["drug_stopped_m6"].sum()),
        len(s) - int(s["drug_stopped_m6"].sum())]
       for _, s in elig_m6.groupby("provider_level")]
chi2_p = stats.chi2_contingency(tab)[1]

print("Drug cessation at 6 months, by provider level")
for _, r in lvl.iterrows():
    print(f"  {r.outcome:20s} {int(r.events):3d}/{int(r.n):3d} = "
          f"{r.pct:.1f}% ({r.lo95:.1f}-{r.hi95:.1f})   sites={int(r.sites)}")
print(f"  chi-squared p = {fmt_p(chi2_p)}")

print("\nWithin-person change at 6 months, by provider level")
prov_chg = []
for c, lab, u in [("d_hba1c_m6", "HbA1c", "%"), ("d_fbs_m6", "FBS", "mg/dL"),
                  ("d_bmi_m6", "BMI", "kg/m²"), ("d_waist_m6", "Waist", "cm")]:
    a = pd.to_numeric(prog.loc[prog["provider_level"] == "community_hospital", c],
                      errors="coerce").dropna()
    b = pd.to_numeric(prog.loc[prog["provider_level"] == "primary_care", c],
                      errors="coerce").dropna()
    pv = stats.ttest_ind(a, b, equal_var=False).pvalue
    prov_chg.append({"measure": lab, "unit": u, "community": a.mean(), "n_comm": len(a),
                     "primary": b.mean(), "n_prim": len(b),
                     "difference": a.mean() - b.mean(), "p": pv})
    print(f"  {lab:6s} community {a.mean():+7.2f} (n={len(a):4d})   "
          f"primary {b.mean():+7.2f} (n={len(b):4d})   p={fmt_p(pv)}")

pd.DataFrame(prov_chg).to_csv(TAB / "table17_by_provider_level.csv", index=False)
lvl.to_csv(TAB / "table17b_cessation_by_provider_level.csv", index=False)

#### The problem: provider level is almost perfectly confounded with the data template

Community hospitals all submitted the Case Record Form. Primary care sites
almost all submitted the รพ.สต. summary workbook. So "provider level" and
"which follow-up column the outcome was read from" are nearly the same
variable — and the two templates do not record the same timepoint.

In [ ]:
print("provider level x source layout (programme cohort):")
print(pd.crosstab(prog["provider_level"], prog["source_family"]).to_string())
print("\nmedication timing tier x provider level:")
print(pd.crosstab(prog["provider_level"], prog["on_any_m6_tier"]).to_string())

#### The test: compare levels *within* the CRF layout only

Two primary-care sites (รพ.สต.หูแร่ and รพ.สต.บ้านโล๊ะจังกะ) used the Case
Record Form rather than the รพ.สต. template. Restricting to CRF-layout patients
holds the template fixed and lets provider level vary on its own.

In [ ]:
crf_only = elig_m6[elig_m6["source_family"].isin(["crf", "crf_pcc"])]
rows = []
for lv, s in crf_only.groupby("provider_level"):
    r = prop_row(lv, s["drug_stopped_m6"])
    r["sites"] = s["facility"].nunique()
    rows.append(r)
within = pd.DataFrame(rows)

tab = [[int(s["drug_stopped_m6"].sum()), len(s) - int(s["drug_stopped_m6"].sum())]
       for _, s in crf_only.groupby("provider_level")]
fisher_p = stats.fisher_exact(tab)[1]

print("Drug cessation at 6 months, CRF-layout patients only")
for _, r in within.iterrows():
    print(f"  {r.outcome:20s} {int(r.events):3d}/{int(r.n):3d} = "
          f"{r.pct:.1f}% ({r.lo95:.1f}-{r.hi95:.1f})   sites={int(r.sites)}")
print(f"  Fisher exact p = {fmt_p(fisher_p)}")
print(f"\n  primary-care sites using the CRF: "
      f"{', '.join(crf_only.loc[crf_only.provider_level=='primary_care','facility'].unique())}")
within.to_csv(TAB / "table17c_provider_level_within_crf.csv", index=False)

In [ ]:
fig, ax = plt.subplots(figsize=(8.6, 4.0))
groups = [("All patients", lvl), ("CRF layout only", within)]
x = np.arange(len(groups)); w = 0.34
for k, (lev, color, label) in enumerate([
        ("community_hospital", dmr_viz.BLUE, "Community hospital"),
        ("primary_care", dmr_viz.ORANGE, "Primary care")]):
    vals, los, his, ns = [], [], [], []
    for _, tb in groups:
        r = tb[tb["outcome"] == lev]
        vals.append(r["pct"].iloc[0] if len(r) else np.nan)
        los.append(r["lo95"].iloc[0] if len(r) else np.nan)
        his.append(r["hi95"].iloc[0] if len(r) else np.nan)
        ns.append(int(r["n"].iloc[0]) if len(r) else 0)
    xp = x + (k - 0.5) * w
    ax.bar(xp, vals, width=w - 0.02, color=color, label=label)
    ax.vlines(xp, los, his, color=dmr_viz.INK_2, lw=1.4)
    for xi, v, hi_, n in zip(xp, vals, his, ns):
        ax.annotate(f"{v:.1f}%  (n={n})", xy=(xi, hi_), xytext=(0, 7),
                    textcoords="offset points", ha="center", fontsize=8.5,
                    color=dmr_viz.INK_2, fontweight="semibold")
ax.set_xticks(x)
ax.set_xticklabels([f"{g}\n(p = {fmt_p(p)})" for (g, _), p in
                    zip(groups, [chi2_p, fisher_p])])
ax.set_ylabel("Off all glucose-lowering drugs\nat 6 months (%)")
ax.set_ylim(0, max(lvl["hi95"].max(), within["hi95"].max()) * 1.35)
ax.legend(loc="upper right")
ax.set_title("The provider-level gap disappears once the data template is held fixed\n"
             "Wilson 95% intervals")
dmr_viz.caption(ax, "Right-hand panel is restricted to sites using the Case Record "
                    "Form, i.e. 4 community hospitals vs 2 primary-care sites.")
fig.tight_layout()
fig.savefig(FIG / "fig11_provider_level.png", bbox_inches="tight")
plt.show()

#### What this means

Holding the template fixed **narrows the gap but does not close it**: 9.4 %
against 17.1 %, Fisher *p* = 0.167. The point estimate on the primary-care side
barely moves, and the interval is very wide — 41 patients and 7 events.

**So this test is inconclusive, not exculpatory.** It neither confirms a
provider-level effect nor rules one out. The confounding with data template is
still real and still unresolved; what has changed is that the within-template
comparison is too small to adjudicate it.

Two observations still cut against reading the headline 19.2 % as a
provider-level effect:

* The per-site table in §5.1 does not split by level — PCC คลองศาลา (community)
  sits at 31 %, above most primary-care sites, while รพ.สต.บ้านโล๊ะจังกะ
  (primary) sits at 0 %.
* The biochemical outcomes, which do **not** depend on the medication column,
  show no provider-level difference except BMI.

But the earlier, stronger claim — that the gap vanishes entirely once the
template is held fixed — **was an artefact of a parser bug in this pipeline**,
not a property of the data. That bug lost every cessation written as
`off` with a short drug name (`off GPZ`) or with no space (`OFFMFM(850)`), which fell disproportionately on รพ.สต.หูแร่, the
larger of the two primary-care sites using the CRF. With it fixed, หูแร่ goes
from 2/22 to 7/27 and the within-template gap reappears.

**The bottom line is unchanged and was always the safer statement: this dataset
cannot separate provider level from data template.** What has changed is that
it now cannot be separated in *either* direction.

#### A site that flips, and why it is not a bug

โรงพยาบาลกงหรา reads **0 / 14** here, but **14 / 14** under the last-observed
definition. Both are correct, and the difference is exactly the clock problem
the anchor exists to expose.

Its CRF records `MFM` before, `MFM` during, and `OFF ยา` in the column labelled
`หลังจบจากโครงการ 6 เดือน`. Those patients came off metformin **6 months after
the programme finished** — not 6 months after they enrolled. At the 6-month
post-enrolment timepoint they were all still on treatment.

Reporting 100 % against a 6-month-since-enrolment outcome would have silently
credited the programme with a result measured on a different clock.

In [ ]:
kr = prog[prog["facility"] == "โรงพยาบาลกงหรา"]
kr[["med_pre_raw", "med_mid_raw", "med_post_raw",
    "on_any_pre", "on_any_m6", "on_any_last",
    "drug_stopped_m6", "drug_stopped"]].head(4)

---
### 5.3 The 2566 survey cohort — a third setting, at 3 months

The `แบบสำรวจ 2566 เครือข่ายกระนวน` export was initially set aside as having no
follow-up medication at all. **That was wrong.** It has two follow-up columns,
`จำนวนเม็ดยาหลัง` and `จำนวนยาฉีดหลัง 3 เดือน`, populated for 213 of 215
patients. It is now analysed as a third cohort at its own 3-month anchor.

Three things make it a *different* measurement from the programme cohort's, and
all three are carried in the data rather than in a footnote:

| | Programme | Registry | Survey 2566 |
|---|---|---|---|
| `med_outcome_basis` | `regimen` — the drug prescribed | `dispensing` + attendance | **`count`** — tablets and injections |
| Anchor | 6 months | not datable | **3 months** |
| Round | current | n/a | **"โครงการ 2"**, Aug 2023 |

It is also a **largely different set of patients** from the 13 รพ.สต. workbooks
covering the same district network — only about 8 % overlap on HN or name — so
it is a separate round, not a duplicate.

In [ ]:
srv = df[df["cohort"] == "survey_2023"].copy()
elig_m3 = df[df["drug_stop_m3_eligible"] == True].copy()

print("Cohorts, with what their medication outcome actually rests on:")
print(df.groupby("cohort").agg(
    patients=("patient_uid", "size"),
    basis=("med_outcome_basis", "first"),
    valid=("med_outcome_valid", "first")).to_string())

print("\nDrug cessation AT 3 MONTHS:")
for coh, s in elig_m3.groupby("cohort"):
    r = prop_row(coh, s["drug_stopped_m3"])
    print(f"  {coh:14s} {int(r['events']):3d}/{int(r['n']):3d} = "
          f"{r['pct']:.1f}% ({r['lo95']:.1f}-{r['hi95']:.1f})   "
          f"basis={s['med_outcome_basis'].iloc[0]}")
print(f"\n  for contrast, programme at 6 months: {pct_ci(elig_m6['drug_stopped_m6'])}")

#### Why 44 % at 3 months and 12 % at 6 months is **not** a time trend

Two checks, because the obvious reading of those numbers is wrong.

**First, the measurement basis is not the problem.** PCC คลองศาลา is the one
programme site with a 3-month medication column, recorded as a *regimen* rather
than a count — and it lands at essentially the same rate as the count-based
survey. So the high 3-month figure is not an artefact of counting tablets.

**Second, the two analysis sets are almost disjoint.** The 3-month programme
set is PCC alone; the 6-month set is dominated by รพ.ท่าวุ้ง and รพ.พิมาย, both
low-cessation sites. The gap is site composition, not elapsed time.

In [ ]:
print("Programme sites contributing to each anchor's cessation set:")
print("  3-month:", prog[prog["drug_stop_m3_eligible"] == True]["facility"].value_counts().to_dict())
print("  6-month:", prog[prog["drug_stop_m6_eligible"] == True]["facility"]
      .value_counts().head(4).to_dict(), "...")

# the one clean within-site, within-patient comparison available
pcc = prog[(prog["facility"] == "PCC คลองศาลา")
           & prog["drug_stop_m3_eligible"] & prog["drug_stop_m6_eligible"]]
print(f"\nPCC คลองศาลา, the same {len(pcc)} patients measured at BOTH anchors:")
print(f"  off all drugs at 3 months: {100*pcc['drug_stopped_m3'].mean():.0f}%")
print(f"  off all drugs at 6 months: {100*pcc['drug_stopped_m6'].mean():.0f}%")
restart = int(((pcc["drug_stopped_m3"] == 1) & (pcc["drug_stopped_m6"] == 0)).sum())
print(f"  stopped at 3 months but back on treatment at 6: {restart}")

Within the same patients at the same site, cessation **falls** between 3 and 6
months — some patients restart. That is a small sample and should not be
over-read, but it is the only within-person evidence in the dataset on whether
deprescribing holds, and it points at *durability* being the thing to measure
rather than the 3-month snapshot.

#### The survey's HbA1c result, and why it does not survive adjustment

The survey cohort's HbA1c fell by **−0.94** points at 3 months — far more than
the programme cohort's −0.27 at 6 months, and far more than the registry's
−0.03 over the same 3-month window. Taken at face value that looks like a
strong result.

It is almost entirely regression to the mean. The survey cohort starts a full
percentage point higher (8.23 vs 7.23), and patients with worse control have
more room to improve. Adjusting for baseline HbA1c removes the difference
completely.

In [ ]:
import statsmodels.formula.api as smf_

anc = df[df["cohort"].isin(["survey_2023", "registry"])].dropna(
    subset=["d_hba1c_m3", "hba1c_pre"]).copy()
anc["survey"] = (anc["cohort"] == "survey_2023").astype(int)

print("Baseline HbA1c:  survey "
      f"{srv['hba1c_pre'].mean():.2f}   registry {reg['hba1c_pre'].mean():.2f}\n")

for label, formula in [("unadjusted", "d_hba1c_m3 ~ survey"),
                       ("adjusted for baseline HbA1c", "d_hba1c_m3 ~ survey + hba1c_pre")]:
    m = smf_.ols(formula, data=anc).fit()
    ci = m.conf_int().loc["survey"]
    print(f"{label:30s} survey vs registry: {m.params['survey']:+.2f} "
          f"(95% CI {ci[0]:+.2f} to {ci[1]:+.2f}), p = {fmt_p(m.pvalues['survey'])}")

print(f"\nn = {len(anc)} ({int(anc['survey'].sum())} survey, "
      f"{int((1-anc['survey']).sum())} registry)")

**−0.92 becomes −0.02 once baseline is accounted for.** This is the clearest
demonstration in the dataset of the regression-to-the-mean warning that opens
this notebook, and it is worth keeping in view when reading *any* of the
single-arm before-and-after figures above: the same mechanism is operating
there, it simply cannot be isolated as neatly without a comparison group.

**What the survey cohort is good for**, then, is not a headline effect size.
It is (a) a second, independent setting in which a high 3-month cessation rate
appears, (b) confirmation that the count-based and regimen-based measurements
agree, and (c) the baseline-adjustment lesson above.

---
### 5.4 Three clinical subgroups

The programme cohort mixes patients in very different clinical situations, and
pooling them hides that. Three subgroups, defined at enrolment:

| | Definition | n | Outcome that makes sense for them |
|---|---|---|---|
| **G1** | Not yet on any glucose-lowering drug | 114 | Did they **avoid starting** one? |
| **G2** | On a drug, HbA1c **> 6.5 %** | 537 | Can control improve enough to deprescribe? |
| **G3** | On a drug, HbA1c **≤ 6.5 %** | 256 | Already at target — can the drug come off? |

88 patients cannot be placed: 73 have no baseline medication record and 15 are
on a drug with no baseline HbA1c.

**Drug cessation is the wrong outcome for G1** — they have nothing to stop. For
them the outcome is inverted: staying drug-free.

In [ ]:
prog = prog.assign(subgroup=np.select(
    [prog["on_any_pre"] == 0,
     (prog["on_any_pre"] == 1) & (prog["hba1c_pre"] > 6.5),
     (prog["on_any_pre"] == 1) & (prog["hba1c_pre"] <= 6.5)],
    ["G1 · drug-naive", "G2 · on drug, HbA1c >6.5", "G3 · on drug, HbA1c ≤6.5"],
    default=None))
sg = prog[prog["subgroup"].notna()]

base = sg.groupby("subgroup").agg(
    n=("subgroup", "size"), sites=("facility", "nunique"),
    age=("age", "mean"), female_pct=("sex", lambda x: 100 * (x == "Female").mean()),
    hba1c=("hba1c_pre", "mean"), fbs=("fbs_pre", "mean"),
    bmi=("bmi_pre", "mean"), waist=("waist_pre", "mean"),
    drug_classes=("n_class_pre", "mean")).round(1)
base.to_csv(TAB / "table18_subgroup_baseline.csv")
base

In [ ]:
# Baseline and change are both taken on the COMPLETE-PAIR set for each measure,
# so that "start + Δ" reconciles with the 6-month mean. Each pair's n is kept.
rows = []
for g, x in sg.groupby("subgroup"):
    r = {"subgroup": g, "n": len(x)}
    for pre, post, lab in [("hba1c_pre", "hba1c_m6", "HbA1c"),
                           ("fbs_pre", "fbs_m6", "FBS"),
                           ("bmi_pre", "bmi_m6", "BMI"),
                           ("waist_pre", "waist_m6", "Waist")]:
        pr = x[[pre, post]].dropna()
        r[lab + " start"] = round(pr[pre].mean(), 2) if len(pr) else np.nan
        r["Δ" + lab] = round((pr[post] - pr[pre]).mean(), 2) if len(pr) else np.nan
        r[lab + " n"] = len(pr)
    for c, lab in [("a1c_lt7_m6", "HbA1c<7% @6m"), ("a1c_lt65_m6", "HbA1c<6.5% @6m"),
                   ("remission_m6", "Remission @6m")]:
        v = pd.to_numeric(x[c], errors="coerce").dropna()
        r[lab] = round(100 * v.mean(), 1) if len(v) else np.nan
    rows.append(r)
sub_out = pd.DataFrame(rows)
sub_out.to_csv(TAB / "table19_subgroup_outcomes.csv", index=False)
sub_out[["subgroup", "n", "HbA1c start", "ΔHbA1c", "BMI start", "ΔBMI",
         "Waist start", "ΔWaist", "HbA1c<6.5% @6m", "Remission @6m"]]

In [ ]:
print("Medication outcome, each subgroup on the outcome that fits it\n")

g1 = sg[(sg["subgroup"] == "G1 · drug-naive") & sg["on_any_m6"].notna()]
stayed = int((g1["on_any_m6"] == 0).sum())
print(f"G1 · drug-naive  (n={len(g1)})")
print(f"   still drug-free at 6 months : {pct_ci(g1['on_any_m6'].eq(0).astype(float))}")
print(f"   had to START a drug         : {100*(g1['on_any_m6']==1).mean():.1f}%\n")

for g in ["G2 · on drug, HbA1c >6.5", "G3 · on drug, HbA1c ≤6.5"]:
    x = sg[(sg["subgroup"] == g) & (sg["drug_stop_m6_eligible"] == True)]
    print(f"{g}  (n={len(x)})")
    print(f"   off all drugs at 6 months   : {pct_ci(x['drug_stopped_m6'])}")

a = sg[(sg["subgroup"].str.startswith("G2")) & (sg["drug_stop_m6_eligible"] == True)]["drug_stopped_m6"]
b = sg[(sg["subgroup"].str.startswith("G3")) & (sg["drug_stop_m6_eligible"] == True)]["drug_stopped_m6"]
tab = [[int(a.sum()), len(a) - int(a.sum())], [int(b.sum()), len(b) - int(b.sum())]]
print(f"\n   G2 vs G3 cessation: chi-squared p = {fmt_p(stats.chi2_contingency(tab)[1])}")
print("   Patients already at target are about twice as likely to be taken off "
      "treatment — which is what a sensible deprescribing protocol should do.")

#### What each subgroup shows

**G1 — not yet on a drug.** The biggest improvements in the whole cohort
(HbA1c −0.85, FBS −39, waist −6.5 cm). But they also start with the *highest*
baseline HbA1c (8.3), so regression to the mean is strongest here and these
figures should be read as the most inflated in the deck. And **36 % had to
start a drug within 6 months** — the programme did not keep them off treatment.
Those who did need one began at HbA1c 10.3 against 7.2 for those who did not,
so baseline severity, not the programme, is what separates them.

**G2 — on a drug, above target.** The largest group and the hardest: HbA1c
falls 0.48, but only 10.5 % come off treatment and just 30 % reach HbA1c < 7 %.

**G3 — on a drug, already at target.** The highest cessation rate (19.0 %) and
much the best control at follow-up (78 % below 7 %). But their mean HbA1c
**rose** by 0.44. Partly a floor effect — they began at 6.1 — and partly what
happens when treatment is withdrawn.

#### A result that will be misread — please read this before quoting it

Within G3, patients who came off their drugs did **better** than those kept on
them. That is a real pattern in the data and it is almost certainly **not**
causal.

In [ ]:
g3 = sg[(sg["subgroup"] == "G3 · on drug, HbA1c ≤6.5")
        & (sg["drug_stop_m6_eligible"] == True)]
rows = []
for lab, x in [("Stopped all drugs", g3[g3["drug_stopped_m6"] == 1]),
               ("Kept on treatment", g3[g3["drug_stopped_m6"] == 0])]:
    r = {"group": lab, "n": len(x)}
    for pre, post, k in [("hba1c_pre", "hba1c_m6", "HbA1c"),
                         ("bmi_pre", "bmi_m6", "BMI"),
                         ("waist_pre", "waist_m6", "Waist")]:
        pr = x[[pre, post]].dropna()
        r[k + " start"] = round(pr[pre].mean(), 2)
        r[k + " @6m"] = round(pr[post].mean(), 2)
        r["Δ" + k] = round((pr[post] - pr[pre]).mean(), 2)
        r[k + " n"] = len(pr)
    r["back above 7%"] = f"{100*(x['hba1c_m6']>=7).mean():.0f}%"
    rows.append(r)
t = pd.DataFrame(rows)

for c, lab in [("d_hba1c_m6", "ΔHbA1c"), ("d_bmi_m6", "ΔBMI"), ("d_waist_m6", "ΔWaist")]:
    a = pd.to_numeric(g3.loc[g3.drug_stopped_m6 == 1, c], errors="coerce").dropna()
    b = pd.to_numeric(g3.loc[g3.drug_stopped_m6 == 0, c], errors="coerce").dropna()
    print(f"  {lab:8s} stopped {a.mean():+6.2f} vs kept {b.mean():+6.2f}   "
          f"p = {fmt_p(stats.ttest_ind(a, b, equal_var=False).pvalue)}")
t.to_csv(TAB / "table20_g3_deprescribing.csv", index=False)
print()
t[["group", "n", "HbA1c start", "HbA1c @6m", "BMI start", "ΔBMI",
   "Waist start", "ΔWaist", "back above 7%"]]

Weight moves the same way and points at a possible mechanism: the patients who
came off treatment held their weight (ΔBMI −0.04, waist −1.8 cm) while those
kept on it gained (ΔBMI +1.06, waist +0.4 cm). **Neither difference reaches
significance** (p = 0.16 and 0.25), so this is a direction, not a finding — but
it is consistent with the reading that clinicians stopped the patients whose
lifestyle change was holding, and it gives a more plausible mechanism than the
drug withdrawal itself.

**This is confounding by indication, not a treatment effect.** Baseline HbA1c
is almost identical in the two rows (6.02 vs 6.08), so the difference is not
about who was sicker on paper. It is about which patients the clinician judged
stable enough to stop — and that judgement used information the dataset does
not contain: adherence, diet, weight trajectory, how long control had been
stable, comorbidity, whether the patient wanted to stop.

Read the right way round: **clinicians selected well.** The patients they chose
to deprescribe overwhelmingly held their control (3 % drifted back above 7 %),
while the ones they judged too precarious to stop drifted anyway (25 %).

Read the wrong way round — "stopping drugs improves control" — it is dangerous
and this design cannot support it. Nothing here compares stopping against
continuing *in the same kind of patient*.

#### Caveat: two subgroups are dominated by one site each

G1 is 69 % รพ.บางระกำ and G3 is 71 % รพ.ท่าวุ้ง. A subgroup difference here
may be a site difference wearing a clinical label — the same confounding as
§5.2, in a different guise.

In [ ]:
conc = sg.groupby("subgroup")["facility"].agg(
    n="size", sites="nunique",
    largest_site=lambda x: x.value_counts().index[0],
    largest_pct=lambda x: round(100 * x.value_counts().iloc[0] / len(x)))
conc.to_csv(TAB / "table21_subgroup_site_concentration.csv")
conc

---
## 6 · Univariable screening

Crude odds ratios for drug cessation, one predictor at a time. This is a
**description of the unadjusted associations**, not a variable-selection step —
the multivariable model in §6 is specified on clinical grounds and on data
availability, not chosen by these *p* values. Selecting covariates by
univariable significance is a known way to produce optimistic, unstable models.

In [ ]:
# the univariable and multivariable models use the TIME-BOUND outcome
elig_m6 = elig_m6.assign(
    female=(elig_m6["sex"] == "Female").astype(float),
    age10=elig_m6["age"] / 10,
    insulin=elig_m6["on_insulin_pre"],
    monotherapy=elig_m6["baseline_monotherapy"],
    outcome=elig_m6["drug_stopped_m6"],
)
elig = elig.assign(
    female=(elig["sex"] == "Female").astype(float),
    age10=elig["age"] / 10,
    insulin=elig["on_insulin_pre"],
    monotherapy=elig["baseline_monotherapy"],
)

UNIVAR = [("age10", "Age, per 10 years"),
          ("female", "Female (vs male)"),
          ("hba1c_pre", "Baseline HbA1c, per 1%"),
          ("fbs_pre", "Baseline FBS, per 10 mg/dL"),
          ("bmi_pre", "Baseline BMI, per 1 kg/m²"),
          ("waist_pre", "Baseline waist, per 1 cm"),
          ("n_class_pre", "Baseline drug classes, per 1"),
          ("insulin", "On insulin at baseline"),
          ("monotherapy", "Baseline monotherapy (≤1 class)"),
          ("d_bmi_m6", "Change in BMI at 6 mo, per 1 kg/m²"),
          ("d_waist_m6", "Change in waist at 6 mo, per 1 cm"),
          ("dm_duration_y", "Diabetes duration, per 1 year"),
          ("n_comorbid", "Number of comorbidities")]

rows = []
for v, lab in UNIVAR:
    d = elig_m6[[v, "outcome", "facility"]].dropna()
    if len(d) < 30 or d["outcome"].nunique() < 2:
        continue
    x = d[v] / 10 if v == "fbs_pre" else d[v]
    dd = d.assign(_x=x)
    try:
        m = smf.gee("outcome ~ _x", groups="facility", data=dd,
                    family=sm.families.Binomial(),
                    cov_struct=sm.cov_struct.Exchangeable()).fit()
    except Exception:
        continue
    ci = m.conf_int().loc["_x"]
    rows.append({"predictor": lab, "n": len(d), "events": int(d["outcome"].sum()),
                 "OR": np.exp(m.params["_x"]),
                 "lo95": np.exp(ci[0]), "hi95": np.exp(ci[1]),
                 "p": m.pvalues["_x"]})

uni = pd.DataFrame(rows)
uni_disp = uni.assign(**{"OR (95% CI)": lambda d:
    d.OR.round(2).astype(str) + " (" + d.lo95.round(2).astype(str)
    + "-" + d.hi95.round(2).astype(str) + ")"},
    p_=lambda d: d.p.map(fmt_p))[["predictor", "n", "events", "OR (95% CI)", "p_"]]
uni_disp.columns = ["Predictor", "n", "Events", "Crude OR (95% CI)", "p"]
uni.to_csv(TAB / "table6_univariable.csv", index=False)
uni_disp

### 7.1 Why the model is specified this way

**The outcome is the time-bound one**: off all glucose-lowering drugs **at 6
months after enrolment** (§3). The last-observed version is refitted in §7.8 as
a sensitivity analysis.

**Clustering.** Patients are nested in sites that differ in protocol,
prescribing culture and how aggressively they deprescribe. Ignoring that gives
standard errors that are too narrow. The primary model is a **GEE logistic
regression with an exchangeable working correlation and facility as the
cluster** — population-averaged odds ratios with cluster-robust standard
errors, which is the right target for a service-evaluation question.

**Two models, deliberately separated.**

* **Model A — baseline determinants.** Only variables known *at enrolment*.
  This is the clinically useful model: on the day a patient enrols, who is
  likely to become drug-free by month 6?
* **Model B — adds 6-month change** in BMI and waist. Weight change happens
  *after* enrolment and lies on the causal path between the programme and the
  outcome. Putting it in Model A would be adjusting for a mediator. It is kept
  separate so "does losing weight accompany cessation?" can be asked without
  contaminating the baseline model.

**What is deliberately left out.** Diabetes duration, benefit scheme and
comorbidity count are not recorded in the รพ.สต. template. Including them would
drop every รพ.สต. patient and silently change the population. They are fitted
separately in §7.5.

**Events per variable** is printed below so the reader can judge overfitting.

In [ ]:
# Baseline insulin is NOT a covariate (2026-11-11): after the rerun no baseline
# insulin user stopped all drugs (0 of 26 in the model set), so the term is not
# estimable (complete separation, as in the replication subset of §7.9) and the
# exchangeable GEE diverges.  Insulin still counts as a class in n_class_pre, and
# `insulin` stays in varsA so the model set is unchanged.  Decided by Percy.
MODEL_A = "outcome ~ age10 + female + hba1c_pre + bmi_pre + fbs10 + n_class_pre"
MODEL_B = MODEL_A + " + d_bmi_m6 + d_waist_m6"
# provider level is a CLUSTER-level covariate: unlike `cohort` (§7.9) it is
# estimable here, because there are several facilities at each level.  It is
# reported with the confounding caveat from §5.2 attached.
MODEL_P = MODEL_A + " + C(provider_level)"

mdat = elig_m6.assign(fbs10=elig_m6["fbs_pre"] / 10)
varsA = ["outcome", "age10", "female", "hba1c_pre", "bmi_pre", "fbs10",
         "n_class_pre", "insulin", "facility"]
varsB = varsA + ["d_bmi_m6", "d_waist_m6"]
dA = mdat.dropna(subset=varsA)
dB = mdat.dropna(subset=varsB)
dP = mdat.dropna(subset=varsA + ["provider_level"])

for nm, d, f in [("Model A (baseline only)", dA, MODEL_A),
                 ("Model B (+ 6-month change)", dB, MODEL_B),
                 ("Model P (+ provider level)", dP, MODEL_P)]:
    k = f.count("+") + 1
    ev = int(d["outcome"].sum())
    print(f"{nm:32s} n={len(d):4d}  events={ev:4d}  sites={d.facility.nunique():3d}  "
          f"covariates={k}  EPV={ev/k:.1f}")

print("\nOutcome: off all glucose-lowering drugs AT 6 MONTHS after enrolment.")
print("Timing tiers in the model set:", tier_mix(dA, "on_any_m6"))
print("Sites per provider level:",
      dP.groupby("provider_level")["facility"].nunique().to_dict())

### 7.2 Collinearity check

Baseline HbA1c and FBS measure the same construct and could be collinear
enough to destabilise both coefficients. Variance inflation factors are checked
before interpreting anything; a VIF above about 5 would be a problem.

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

X = sm.add_constant(dA[["age10", "female", "hba1c_pre", "bmi_pre", "fbs10",
                        "n_class_pre"]].astype(float))
vif = pd.DataFrame({
    "variable": X.columns,
    "VIF": [variance_inflation_factor(X.values, i) for i in range(X.shape[1])],
}).query("variable != 'const'").round(2)
print(vif.to_string(index=False))
print(f"\ncorr(HbA1c, FBS) = {dA['hba1c_pre'].corr(dA['fbs10']):.2f}")

### 7.3 Fit the models

In [ ]:
def fit_gee(formula, data):
    return smf.gee(formula, groups="facility", data=data,
                   family=sm.families.Binomial(),
                   cov_struct=sm.cov_struct.Exchangeable()).fit()


def tidy_or(model, labels=None, drop_intercept=True):
    '''Odds ratios with 95 % CI from a fitted logit-link model.'''
    ci = model.conf_int()
    t = pd.DataFrame({"term": model.params.index,
                      "OR": np.exp(model.params.values),
                      "lo95": np.exp(ci[0].values),
                      "hi95": np.exp(ci[1].values),
                      "p": model.pvalues.values})
    if drop_intercept:
        t = t[t["term"] != "Intercept"]
    if labels:
        t["term"] = t["term"].map(labels).fillna(t["term"])
    return t.reset_index(drop=True)


LABELS = {
    "age10": "Age, per 10 years", "female": "Female (vs male)",
    "hba1c_pre": "Baseline HbA1c, per 1%", "bmi_pre": "Baseline BMI, per 1 kg/m²",
    "fbs10": "Baseline FBS, per 10 mg/dL",
    "n_class_pre": "Baseline drug classes, per 1",
    "insulin": "On insulin at baseline",
    "d_bmi": "Change in BMI, per 1 kg/m²",
    "d_waist": "Change in waist, per 1 cm",
    "d_bmi_m6": "Change in BMI at 6 mo, per 1 kg/m²",
    "d_waist_m6": "Change in waist at 6 mo, per 1 cm",
    "dm_duration_y": "Diabetes duration, per 1 year",
    "n_comorbid": "Comorbidity count, per 1",
    "C(scheme)[T.CSMBS]": "Benefit scheme: CSMBS (vs UCS)",
    "C(scheme)[T.SSS]": "Benefit scheme: SSS (vs UCS)",
    "C(scheme)[T.Other]": "Benefit scheme: Other (vs UCS)",
}

gA = fit_gee(MODEL_A, dA)
gB = fit_gee(MODEL_B, dB)

tA = tidy_or(gA, LABELS); tA["model"] = "A · baseline only"
tB = tidy_or(gB, LABELS); tB["model"] = "B · + 6-month change"
res = pd.concat([tA, tB], ignore_index=True)

print(f"Model A exchangeable within-site correlation: "
      f"{gA.cov_struct.dep_params:.3f}")
print(f"Model B exchangeable within-site correlation: "
      f"{gB.cov_struct.dep_params:.3f}\n")

disp = res.assign(**{"aOR (95% CI)": lambda d:
    d.OR.round(2).astype(str) + " (" + d.lo95.round(2).astype(str)
    + "–" + d.hi95.round(2).astype(str) + ")"},
    p_=lambda d: d.p.map(fmt_p))[["model", "term", "aOR (95% CI)", "p_"]]
disp.columns = ["Model", "Predictor", "Adjusted OR (95% CI)", "p"]
res.to_csv(TAB / "table7_multivariable_gee.csv", index=False)
disp

#### Model P — adding provider level

Unlike `cohort` in §7.9, provider level **is** estimable alongside facility
clustering, because there are several facilities at each level (4 community,
15 primary in the model set) rather than one.

**Read the coefficient with §5.2 open beside it.** Provider level is almost
perfectly confounded with the data template, and the within-template test there
shows the gap disappearing. The odds ratio below is reported for completeness
and is *not* evidence that primary care deprescribes more successfully.

In [ ]:
gP = fit_gee(MODEL_P, dP)
tP_ = tidy_or(gP, {**LABELS,
                   "C(provider_level)[T.primary_care]":
                       "Primary care (vs community hospital)"})
tP_.to_csv(TAB / "table7b_model_provider_level.csv", index=False)

print(f"n={len(dP)}, events={int(dP['outcome'].sum())}, "
      f"sites={dP['facility'].nunique()} "
      f"({dP.groupby('provider_level')['facility'].nunique().to_dict()})\n")
tP_.assign(**{"aOR (95% CI)": lambda d: d.OR.round(2).astype(str) + " ("
              + d.lo95.round(2).astype(str) + "-" + d.hi95.round(2).astype(str) + ")"},
           p_=lambda d: d.p.map(fmt_p))[["term", "aOR (95% CI)", "p_"]]

The adjusted odds ratio for primary care is large, but with only **4 community
hospital clusters** the cluster-robust standard error for a cluster-level
covariate is fragile, and the confounding with template is unresolved. Treat
this row as a description of the data, not a finding about provider levels.

The within-site correlation is the clustering the model is absorbing. A value
meaningfully above zero means patients in the same site resemble each other on
this outcome beyond what their individual characteristics explain — i.e. *which
site you attend* independently predicts whether you come off your drugs. An
ordinary logistic regression would have reported standard errors too narrow to
be believed.

### 7.4 Forest plot

In [ ]:
plot_df = res[res["model"] == "A · baseline only"].copy()
plot_df = plot_df.iloc[::-1].reset_index(drop=True)

fig, ax = plt.subplots(figsize=(8.8, 0.52 * len(plot_df) + 2.0))
y = np.arange(len(plot_df))
sig = plot_df["p"] < 0.05
ax.hlines(y, plot_df["lo95"], plot_df["hi95"], color=dmr_viz.AXIS, lw=2.2)
ax.plot(plot_df.loc[~sig, "OR"], y[~sig.values], "o", color=dmr_viz.MUTED,
        markersize=8, markeredgecolor=dmr_viz.SURFACE, markeredgewidth=2, ls="none")
ax.plot(plot_df.loc[sig, "OR"], y[sig.values], "o", color=dmr_viz.BLUE,
        markersize=9, markeredgecolor=dmr_viz.SURFACE, markeredgewidth=2, ls="none")
ax.axvline(1, color=dmr_viz.INK_2, lw=1.2, ls="--")
ax.set_xscale("log")
ticks = [t for t in (0.05, 0.1, 0.25, 0.5, 1, 2, 4, 8)
         if plot_df["lo95"].min() / 1.5 <= t <= plot_df["hi95"].max() * 1.5]
ax.set_xticks(ticks)
ax.set_xticklabels([f"{t:g}" for t in ticks])
ax.minorticks_off()
ax.set_xlabel("Adjusted odds ratio for stopping all glucose-lowering drugs (log scale)")
ax.set_yticks(y)
ax.set_yticklabels(plot_df["term"], fontsize=9)
ax.grid(axis="y", visible=False)
for yi, r in plot_df.iterrows():
    ax.annotate(f"{r.OR:.2f} ({r.lo95:.2f}–{r.hi95:.2f})  p={fmt_p(r.p)}",
                xy=(ax.get_xlim()[1], yi), xytext=(8, 0),
                textcoords="offset points", va="center", fontsize=8.5,
                color=dmr_viz.INK_2 if r.p < 0.05 else dmr_viz.MUTED,
                annotation_clip=False)
ax.annotate("OR below 1: less likely to stop        OR above 1: more likely to stop",
            xy=(0.5, -0.155), xycoords="axes fraction", ha="center",
            fontsize=8.5, color=dmr_viz.MUTED)
ax.set_title("Determinants of drug cessation — Model A (baseline variables only)\n"
             f"GEE logistic regression, exchangeable correlation, clustered by facility; "
             f"n={len(dA)}, {int(dA['drug_stopped'].sum())} events, "
             f"{dA['facility'].nunique()} sites")
fig.tight_layout()
fig.savefig(FIG / "fig05_forest_determinants.png", bbox_inches="tight")
plt.show()

### 7.5 Extended model — restricted subset

Adds diabetes duration, benefit scheme and comorbidity count. **This model is
fitted only on patients from the hospital-based layouts**, because the รพ.สต.
template records none of these fields. That subset is a different case mix, not
a random sample, so its coefficients are not directly comparable with Model A
and are reported as exploratory.

In [ ]:
MODEL_C = (MODEL_A + " + dm_duration_y + n_comorbid + C(scheme)")
varsC = varsA + ["dm_duration_y", "n_comorbid", "scheme"]
dC = mdat.dropna(subset=varsC)
dC = dC[dC["scheme"].isin(dC["scheme"].value_counts()[lambda s: s >= 10].index)]
dC["scheme"] = pd.Categorical(
    dC["scheme"], categories=["UCS"] + [c for c in dC["scheme"].unique() if c != "UCS"])

print(f"n={len(dC)}  events={int(dC['outcome'].sum())}  "
      f"sites={dC['facility'].nunique()}  (vs Model A n={len(dA)}, "
      f"{dA['facility'].nunique()} sites)")
print("\nsource layouts represented:")
print(dC["source_family"].value_counts().to_string())

gC = fit_gee(MODEL_C, dC)
tC = tidy_or(gC, LABELS)
tC.to_csv(TAB / "table8_extended_model.csv", index=False)
tC.assign(**{"aOR (95% CI)": lambda d: d.OR.round(2).astype(str) + " ("
             + d.lo95.round(2).astype(str) + "-" + d.hi95.round(2).astype(str) + ")"},
          p_=lambda d: d.p.map(fmt_p))[["term", "aOR (95% CI)", "p_"]]

### 7.6 Between-site variance — random-intercept model

The GEE gives population-averaged effects but not a variance component. A
Bayesian logistic mixed model with a facility random intercept is fitted to
quantify how much of the variation sits *between sites*, expressed as the
latent-scale intraclass correlation
$\mathrm{ICC} = \sigma^2_{site} / (\sigma^2_{site} + \pi^2/3)$.

In [ ]:
from statsmodels.genmod.bayes_mixed_glm import BinomialBayesMixedGLM

bm = BinomialBayesMixedGLM.from_formula(
    MODEL_A, {"site": "0 + C(facility)"}, dA).fit_vb()

sd_site = float(np.exp(bm.vcp_mean[0]))
icc = sd_site**2 / (sd_site**2 + np.pi**2 / 3)
print(f"between-site SD (log-odds scale): {sd_site:.2f}")
print(f"latent-scale ICC                : {icc:.3f}  "
      f"({100*icc:.1f}% of variation is between sites)")
print(f"median odds ratio between two random sites: "
      f"{np.exp(np.sqrt(2) * sd_site * stats.norm.ppf(0.75)):.2f}")

cmp = pd.DataFrame({
    "term": [LABELS.get(t, t) for t in gA.params.index if t != "Intercept"],
    "GEE (population-averaged) OR": [np.exp(gA.params[t]) for t in gA.params.index
                                      if t != "Intercept"],
})
idx = {n: i for i, n in enumerate(bm.model.exog_names)}
cmp["Mixed (site-specific) OR"] = [
    np.exp(bm.fe_mean[idx[t]]) for t in gA.params.index if t != "Intercept"]
cmp.round(2)

The two columns answer different questions and are expected to differ: the GEE
odds ratio is the average effect across the whole programme, the mixed-model
odds ratio is the effect *within a given site*. Site-specific effects are
normally further from 1 than population-averaged ones, and that pattern is
what should be seen here.

### 7.7 Model performance and calibration

Discrimination (C-statistic), calibration (Hosmer–Lemeshow, calibration plot),
and an honest internal-validity check by bootstrap optimism correction. The
apparent C-statistic is always optimistic on the data the model was fitted to;
the optimism-corrected value is the one to quote.

In [ ]:
def c_statistic(y, p):
    '''Area under the ROC curve, computed from the rank statistic.'''
    y = np.asarray(y, dtype=int); p = np.asarray(p, float)
    n1, n0 = y.sum(), (1 - y).sum()
    if n1 == 0 or n0 == 0:
        return np.nan
    r = stats.rankdata(p)
    return (r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)


def hosmer_lemeshow(y, p, g=10):
    '''Hosmer–Lemeshow goodness-of-fit over deciles of predicted risk.'''
    d = pd.DataFrame({"y": np.asarray(y, int), "p": np.asarray(p, float)})
    d["grp"] = pd.qcut(d["p"].rank(method="first"), g, labels=False)
    agg = d.groupby("grp").agg(obs=("y", "sum"), exp=("p", "sum"), n=("y", "size"))
    chi2 = (((agg["obs"] - agg["exp"]) ** 2)
            / (agg["exp"] * (1 - agg["exp"] / agg["n"]))).sum()
    return float(chi2), int(g - 2), float(stats.chi2.sf(chi2, g - 2)), agg


y = dA["outcome"].to_numpy(int)
p_hat = gA.fittedvalues.to_numpy(float)

c_app = c_statistic(y, p_hat)
chi2, dof, p_hl, hl_tab = hosmer_lemeshow(y, p_hat)

# bootstrap optimism correction (Harrell): refit on each resample, measure the
# drop in C when the resample model is applied to the original data
rng = np.random.default_rng(20260920)
opt = []
for _ in range(200):
    idx = rng.integers(0, len(dA), len(dA))
    boot = dA.iloc[idx]
    if boot["drug_stopped"].nunique() < 2:
        continue
    try:
        mb = smf.glm(MODEL_A, data=boot, family=sm.families.Binomial()).fit()
        c_boot = c_statistic(boot["drug_stopped"], mb.fittedvalues)
        c_orig = c_statistic(y, mb.predict(dA))
        opt.append(c_boot - c_orig)
    except Exception:
        continue

print(f"C-statistic (apparent)            : {c_app:.3f}")
print(f"C-statistic (optimism-corrected)  : {c_app - np.mean(opt):.3f}   "
      f"[{len(opt)} bootstrap replicates]")
verdict = ("no evidence of miscalibration" if p_hl > 0.05
           else "EVIDENCE OF MISCALIBRATION — see the note below")
print(f"Hosmer\u2013Lemeshow                   : \u03c7\u00b2({dof}) = {chi2:.2f}, "
      f"p = {fmt_p(p_hl)}")
print(f"                                    \u2192 {verdict}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.1))

# (a) ROC
ax = axes[0]
order = np.argsort(-p_hat)
tpr = np.cumsum(y[order]) / y.sum()
fpr = np.cumsum(1 - y[order]) / (1 - y).sum()
ax.plot(np.r_[0, fpr], np.r_[0, tpr], color=dmr_viz.BLUE)
ax.plot([0, 1], [0, 1], color=dmr_viz.MUTED, lw=1.2, ls="--")
ax.set_xlabel("False-positive rate"); ax.set_ylabel("True-positive rate")
ax.set_title("(a) Discrimination")
ax.annotate(f"C = {c_app:.3f} apparent\nC = {c_app-np.mean(opt):.3f} corrected",
            xy=(0.58, 0.14), xycoords="axes fraction", fontsize=9,
            color=dmr_viz.INK_2)

# (b) calibration
ax = axes[1]
hl = hl_tab.assign(obs_rate=lambda d: d["obs"] / d["n"],
                   exp_rate=lambda d: d["exp"] / d["n"])
hi = hl[["obs_rate", "exp_rate"]].to_numpy().max() * 1.1
ax.plot([0, hi], [0, hi], color=dmr_viz.MUTED, lw=1.2, ls="--")
ax.plot(hl["exp_rate"], hl["obs_rate"], "o-", color=dmr_viz.BLUE,
        markeredgecolor=dmr_viz.SURFACE, markeredgewidth=2)
ax.set_xlabel("Predicted probability (decile mean)")
ax.set_ylabel("Observed proportion")
ax.set_title("(b) Calibration by decile of predicted risk")
ax.annotate("perfect calibration", xy=(0.52, 0.60), xycoords="axes fraction",
            fontsize=8.5, color=dmr_viz.MUTED, rotation=32)
fig.tight_layout()
fig.savefig(FIG / "fig06_model_performance.png", bbox_inches="tight")
plt.show()

**What the calibration plot is telling us.** Discrimination is good — the model
ranks patients by their chance of stopping well above chance. But the
Hosmer–Lemeshow test rejects, and panel (b) shows why: in the highest deciles
of predicted risk the model **predicts more cessations than actually occur**,
and the points sit consistently below the diagonal.

This is not a coding error, it is the clustering showing through. The GEE
produces *population-averaged* predictions, while §7.6 puts nearly 40 % of the
variance between sites. A patient with favourable baseline characteristics at a
site that rarely deprescribes does not have the programme-average probability of
stopping — they have their site's probability, which is much lower.

The practical consequence: **these coefficients are usable for understanding
which factors matter, but the model should not be used as an individual risk
calculator** without a site term. That, again, points at the organisational
finding rather than the clinical one.

### 7.8 Sensitivity analyses

Four checks on whether the headline finding depends on a debatable choice:

1. **Outcome definition** — refit with the old last-observed outcome. If the
   coefficients hold, the time anchor changed the *rate* but not the
   *determinants*.
2. **Timing tier** — restrict to patients whose 6-month timing is `measured` or
   `labelled`. Medication has no `measured` tier anywhere (§3), so this is a
   small check, and the cell reports honestly when it cannot be fitted.
3. **Excluding the largest site** — one site contributes a quarter of the
   cohort.
4. **Multiple imputation** — chained equations for the baseline covariates.
   Diabetes duration is *not* imputed: it is missing by template design for
   whole sites, which is a structural gap, not a pattern imputation can fill.

In [ ]:
sens = []

def add_sens(label, model, n, ev):
    t = tidy_or(model, LABELS)
    t["analysis"] = label; t["n"] = n; t["events"] = ev
    sens.append(t)

add_sens("Primary: cessation at 6 months", gA, len(dA), int(dA["outcome"].sum()))

# 1. the previous definition - whatever regimen was recorded last
d_last = elig.assign(fbs10=elig["fbs_pre"] / 10,
                     outcome=elig["drug_stopped"]).dropna(subset=varsA)
add_sens("Outcome = last observed", fit_gee(MODEL_A, d_last), len(d_last),
         int(d_last["outcome"].sum()))

# 2. restricted to patients whose 6-month timing is measured or labelled
d_tier = dA[dA["on_any_m6_tier"].isin(["measured", "labelled"])]
if len(d_tier) > 40 and d_tier["outcome"].nunique() > 1:
    try:
        add_sens("Timing measured/labelled only",
                 smf.glm(MODEL_A, data=d_tier, family=sm.families.Binomial()).fit(cov_type="HC1"),
                 len(d_tier), int(d_tier["outcome"].sum()))
    except Exception as exc:
        print(f"timing-restricted model not estimable: {exc}")
else:
    print(f"Timing measured/labelled only: n={len(d_tier)}, "
          f"events={int(d_tier['outcome'].sum()) if len(d_tier) else 0} "
          "- too few to fit; see §3 (medication has no `measured` tier at all).")

# 3. drop the largest site
big = dA["facility"].value_counts().idxmax()
d_nb = dA[dA["facility"] != big]
add_sens(f"Excluding {big}", fit_gee(MODEL_A, d_nb), len(d_nb),
         int(d_nb["outcome"].sum()))

# 4. multiple imputation of the baseline covariates
from statsmodels.imputation import mice

mi_vars = ["outcome", "age10", "female", "hba1c_pre", "bmi_pre", "fbs10",
           "n_class_pre", "insulin"]
mi_src = mdat.dropna(subset=["outcome", "facility"])[mi_vars + ["facility"]]
np.random.seed(20261112)  # MICE draws from numpy's global RNG; fixed so table9 reproduces
imp = mice.MICEData(mi_src.drop(columns="facility").astype(float))
params, covs = [], []
for _ in range(10):
    imp.update_all()
    filled = imp.data.copy()
    filled["facility"] = mi_src["facility"].to_numpy()
    filled["outcome"] = mi_src["outcome"].to_numpy()      # never imputed
    m = fit_gee(MODEL_A, filled)
    params.append(m.params); covs.append(np.diag(m.cov_params()))

P_ = pd.concat(params, axis=1)
qbar = P_.mean(axis=1); ubar = np.mean(covs, axis=0); B = P_.var(axis=1, ddof=1)
T = ubar + (1 + 1 / len(params)) * B          # Rubin's rules
se = np.sqrt(T)
mi_tab = pd.DataFrame({
    "term": [LABELS.get(t, t) for t in qbar.index],
    "OR": np.exp(qbar.values),
    "lo95": np.exp(qbar.values - 1.96 * se.values),
    "hi95": np.exp(qbar.values + 1.96 * se.values),
    "p": 2 * stats.norm.sf(np.abs(qbar.values / se.values))})
mi_tab = mi_tab[mi_tab["term"] != "Intercept"]
mi_tab["analysis"] = "Multiple imputation (10 sets)"
mi_tab["n"] = len(mi_src); mi_tab["events"] = int(mi_src["outcome"].sum())
sens.append(mi_tab)

sens_tab = pd.concat(sens, ignore_index=True)
sens_tab.to_csv(TAB / "table9_sensitivity.csv", index=False)

pivot = sens_tab.assign(cell=lambda d: d.OR.round(2).astype(str) + " ("
                        + d.lo95.round(2).astype(str) + "-"
                        + d.hi95.round(2).astype(str) + ")").pivot(
    index="term", columns="analysis", values="cell")
print(sens_tab.groupby("analysis")[["n", "events"]].first().to_string())
pivot

The finding is robust if the key odds ratios keep the same direction and rough
magnitude across all four columns. Where an interval widens under the strict
outcome, that is loss of sample size rather than a contradicted result.

### 7.9 Does it replicate outside the programme? — and why the two cohorts cannot be pooled

The programme cohort and the registry are two different settings. If the same
baseline factors predict cessation in both, that is a genuine replication and
the finding is much harder to dismiss as an artefact of one programme's
selection rules.

**Pooling them into one model is not an option, and the reason is structural.**
The registry is a *single facility*. Cohort is therefore perfectly nested
inside facility, so a model containing both a cohort term and facility
clustering has nothing left to identify the cohort effect from. The cell below
demonstrates that rather than asserting it — the coefficients diverge to 0 and
∞ and the standard errors come back undefined.

In [ ]:
pooled = df[df["drug_stop_eligible"] == True].assign(
    female=lambda d: (d["sex"] == "Female").astype(float),
    age10=lambda d: d["age"] / 10,
    fbs10=lambda d: d["fbs_pre"] / 10,
    insulin=lambda d: d["on_insulin_pre"],
    outcome=lambda d: d["drug_stopped"],
).dropna(subset=varsA + ["dm_duration_y"])

print("facilities contributing to each cohort:")
print(pooled.groupby("cohort")["facility"].nunique().to_string())
print("\n-> the registry is one facility, so `cohort` is nested inside `facility`.\n")

bad = fit_gee(MODEL_A + " + dm_duration_y + C(cohort)", pooled)
print("pooled model with BOTH a cohort term and facility clustering:")
print(pd.DataFrame({"OR": np.exp(bad.params), "std err": bad.bse,
                    "p": bad.pvalues}).round(3).to_string())
print("\nUnidentifiable, as expected. The two cohorts are modelled separately below.")

So each cohort gets its own model. The programme cohort keeps facility
clustering; the registry, being one site, needs none and uses an ordinary
logistic regression with heteroskedasticity-robust standard errors.

**The two sides run on different clocks, and that has to be said plainly.**
The registry's medication sheet has no dates, so a 6-month cessation outcome
does not exist there (§3). This comparison therefore uses the **last-observed**
outcome on both sides — 6–12 months in the programme, a median of 28 months in
the registry. It is a replication of *which factors predict cessation*, not a
comparison of cessation *rates*, and the rates should not be read off it.

Both models add **diabetes duration**, which the บ้านตาขุน export now supplies
for 1,458 patients. Requiring it restricts the programme side to its **six
hospital-based sites** — a narrower comparison than §7.3, but a fairer one,
since the registry is itself a hospital.

**One term has to be dropped: baseline insulin use.** In the programme subset
only 10 patients were on insulin and *none* stopped, which is complete
separation — the estimate diverges and the software returns an odds ratio of 0
with a meaningless interval.

In [ ]:
# NOTE: this section uses the LAST-OBSERVED outcome on both sides, because the
# registry has no dated medication data and so no 6-month cessation outcome.
rep_vars_full = ["drug_stopped", "age10", "female", "hba1c_pre", "bmi_pre",
                 "fbs10", "n_class_pre", "insulin", "facility", "dm_duration_y"]

base_rep = df[df["drug_stop_eligible"] == True].assign(
    female=lambda d: (d["sex"] == "Female").astype(float),
    age10=lambda d: d["age"] / 10,
    fbs10=lambda d: d["fbs_pre"] / 10,
    insulin=lambda d: d["on_insulin_pre"])

d_prog_rep = base_rep[base_rep["cohort"] == "program"].dropna(subset=rep_vars_full)
d_reg_rep = base_rep[base_rep["cohort"] == "registry"].dropna(subset=rep_vars_full)

print("Baseline insulin use x drug cessation, programme subset:")
print(pd.crosstab(d_prog_rep["insulin"], d_prog_rep["drug_stopped"],
                  rownames=["on insulin"], colnames=["stopped"]).to_string())
print("\n-> a zero cell: complete separation. The insulin term is not estimable "
      "here and is dropped from both models so they stay comparable.\n")

FORM_REP = ("drug_stopped ~ age10 + female + hba1c_pre + bmi_pre + fbs10 "
            "+ n_class_pre + dm_duration_y")

g_prog = smf.glm(FORM_REP, data=d_prog_rep, family=sm.families.Binomial()).fit(
    cov_type="cluster", cov_kwds={"groups": d_prog_rep["facility"]})
g_reg = smf.glm(FORM_REP, data=d_reg_rep, family=sm.families.Binomial()).fit(cov_type="HC1")

tP = tidy_or(g_prog, LABELS); tP["cohort"] = "Programme"
tR = tidy_or(g_reg, LABELS);  tR["cohort"] = "Registry"
rep = pd.concat([tP, tR], ignore_index=True)
rep.to_csv(TAB / "table12_replication_by_cohort.csv", index=False)

print(f"Programme: n={len(d_prog_rep)}, events={int(d_prog_rep['drug_stopped'].sum())}, "
      f"sites={d_prog_rep['facility'].nunique()}  [logistic, facility-clustered SEs]")
print(f"Registry : n={len(d_reg_rep)}, events={int(d_reg_rep['drug_stopped'].sum())}, "
      f"sites=1  [logistic, HC1 robust SEs]")
print("Outcome on both sides: LAST-OBSERVED cessation (different follow-up lengths).\n")

rep.assign(**{"aOR (95% CI)": lambda d: d.OR.round(2).astype(str) + " ("
              + d.lo95.round(2).astype(str) + "-" + d.hi95.round(2).astype(str) + ")"},
           p_=lambda d: d.p.map(fmt_p)).pivot(
    index="term", columns="cohort", values="aOR (95% CI)")

In [ ]:
# side-by-side forest: do the two settings agree?
terms = [t for t in tP["term"] if t in set(tR["term"])]
fig, ax = plt.subplots(figsize=(9.2, 0.62 * len(terms) + 2.4))
y = np.arange(len(terms))[::-1]
off = 0.17
lo_all, hi_all = [], []
for k, (nm, tt, color) in enumerate([("Programme", tP, dmr_viz.BLUE),
                                     ("Registry", tR, dmr_viz.ORANGE)]):
    tt = tt.set_index("term").reindex(terms)
    lo_all += list(tt["lo95"]); hi_all += list(tt["hi95"])
    yy = y + (off if k == 0 else -off)
    ax.hlines(yy, tt["lo95"], tt["hi95"], color=color, lw=2.0, alpha=0.8)
    ax.plot(tt["OR"], yy, "o", color=color, markersize=7.5,
            markeredgecolor=dmr_viz.SURFACE, markeredgewidth=1.6, ls="none",
            label=nm)
ax.axvline(1, color=dmr_viz.INK_2, lw=1.2, ls="--")
ax.set_xscale("log")
ax.set_xlim(min(lo_all) / 1.6, max(hi_all) * 1.6)   # keep every interval inside
ticks = [t for t in (0.005, 0.01, 0.05, 0.1, 0.25, 0.5, 1, 2, 4, 8)
         if ax.get_xlim()[0] <= t <= ax.get_xlim()[1]]
ax.set_xticks(ticks); ax.set_xticklabels([f"{t:g}" for t in ticks])
ax.minorticks_off()
ax.set_yticks(y); ax.set_yticklabels(terms, fontsize=9)
ax.grid(axis="y", visible=False)
ax.legend(loc="lower right")
ax.set_xlabel("Adjusted odds ratio for stopping all glucose-lowering drugs (log scale)")
ax.set_title(
    "Same model, two settings\n"
    f"Programme, hospital sites only (n={len(d_prog_rep)}, "
    f"{d_prog_rep['facility'].nunique()} sites)  vs  "
    f"รพ.บ้านตาขุน register (n={len(d_reg_rep)}, 1 site)")
dmr_viz.caption(ax, "Insulin use omitted: complete separation in the programme "
                    "subset (10 patients on insulin, 0 stopped).")
fig.tight_layout()
fig.savefig(FIG / "fig09_replication.png", bbox_inches="tight")
plt.show()

**Read the two sets of intervals for agreement in direction, not for identical
magnitude.** The settings differ in follow-up length, case mix and how
deprescribing is decided, so the point estimates should not coincide.

The signal to look for is whether **baseline drug burden** and **baseline
HbA1c** point the same way in both. They do — more drug classes and higher
HbA1c at baseline predict *not* stopping in the programme and in a register
that was never part of it. An association that survives a change of setting is
much less likely to be an artefact of one programme's selection rule.

Note also where they *disagree*: baseline fasting glucose predicts cessation in
the programme but not in the register. That is consistent with FBS being part
of how programme clinicians pick candidates, rather than a property of the
patient — which is a useful thing to know about the programme model, and a
caution against reading these coefficients as physiology.

---
## 8 · Cost — what deprescribing does to spending

The บ้านตาขุน export is the only workbook with costs, recorded per visit as
`date | ค่า Lab | ค่ายา`. Each patient's figures below are the **mean per
visit**, before and after the index date, so a patient who simply attended more
often does not look more expensive.

This matters for a service evaluation in a way the clinical outcomes do not:
stopping a drug removes a recurring cost, but it usually buys more monitoring.
Both sides are visible here.

In [ ]:
cost_rows = []
for lab_, pre_, post_ in [("Drug cost per visit (฿)", "drugcost_pre", "drugcost_post"),
                          ("Lab cost per visit (฿)", "labcost_pre", "labcost_post")]:
    r = paired_change(reg, pre_, post_, lab_, "฿")
    if r:
        cost_rows.append(r)
cost_tab = pd.DataFrame(cost_rows)
cost_tab.to_csv(TAB / "table13_cost_overall.csv", index=False)

cost_tab.assign(
    pre=lambda d: d.pre_mean.round(0), post=lambda d: d.post_mean.round(0),
    change=lambda d: d.mean_change.round(0).astype(str) + " ("
                     + d.lo95.round(0).astype(str) + " to " + d.hi95.round(0).astype(str) + ")",
    p=lambda d: d.p_wilcoxon.map(fmt_p))[
    ["measure", "n_pairs", "pre", "post", "change", "p"]].rename(columns={
    "measure": "Measure", "n_pairs": "n pairs", "pre": "Before (mean)",
    "post": "After (mean)", "change": "Change (95% CI)", "p": "p (Wilcoxon)"})

Overall spending did not fall — which is unsurprising, because most patients
did not stop their drugs. The question is what happened **to the patients who
did**.

In [ ]:
cs = elig_reg.dropna(subset=["drug_stopped"]).copy()
cs["group"] = np.where(cs["drug_stopped"] == 1, "Stopped all drugs", "Still on drugs")

by_grp = cs.groupby("group").agg(
    n=("group", "size"),
    drug_pre=("drugcost_pre", "mean"), drug_post=("drugcost_post", "mean"),
    drug_change=("d_drugcost", "mean"),
    lab_pre=("labcost_pre", "mean"), lab_post=("labcost_post", "mean"),
    lab_change=("d_labcost", "mean"),
).round(0)
by_grp["net_change"] = (by_grp["drug_change"] + by_grp["lab_change"]).round(0)

a = cs.loc[cs["drug_stopped"] == 1, "d_drugcost"].dropna()
b = cs.loc[cs["drug_stopped"] == 0, "d_drugcost"].dropna()
u = stats.mannwhitneyu(a, b)
print(f"Change in drug cost per visit, stoppers vs non-stoppers: "
      f"{a.mean():+.0f} vs {b.mean():+.0f} THB, Mann-Whitney p = {fmt_p(u.pvalue)}")
by_grp.to_csv(TAB / "table14_cost_by_cessation.csv")
by_grp

In [ ]:
fig, ax = plt.subplots(figsize=(8.6, 4.2))
groups = ["Still on drugs", "Stopped all drugs"]
x = np.arange(len(groups)); w = 0.34
drug = by_grp.reindex(groups)["drug_change"]
lab = by_grp.reindex(groups)["lab_change"]
ax.bar(x - w/2, drug, width=w - 0.02, color=dmr_viz.BLUE, label="Drug cost")
ax.bar(x + w/2, lab, width=w - 0.02, color=dmr_viz.ORANGE, label="Lab cost")
ax.axhline(0, color=dmr_viz.AXIS, lw=1)
for xi, (dv, lv) in enumerate(zip(drug, lab)):
    for dx, v in [(-w/2, dv), (w/2, lv)]:
        ax.annotate(f"{v:+,.0f}", xy=(xi + dx, v),
                    xytext=(0, 5 if v >= 0 else -13), textcoords="offset points",
                    ha="center", fontsize=9, color=dmr_viz.INK_2, fontweight="semibold")
ax.set_xticks(x)
ax.set_xticklabels([f"{g}\n(n = {int(by_grp.loc[g, 'n'])})" for g in groups])
ax.set_ylabel("Change in mean cost per visit (฿)")
ax.legend(loc="upper left")
ax.set_title("Deprescribing shifts spending from pharmacy to laboratory\n"
             "รพ.บ้านตาขุน register, cost per visit before vs after the index date")
fig.tight_layout()
fig.savefig(FIG / "fig10_cost_by_cessation.png", bbox_inches="tight")
plt.show()

**The trade is visible and it is not free.** Patients who came off their drugs
saw pharmacy spending per visit fall substantially, while laboratory spending
rose — they are monitored more closely, which is exactly what a responsible
deprescribing protocol should produce. The `net_change` column in the table
above is the honest bottom line, and it is much smaller than the drug saving
alone would suggest.

Two caveats. `ค่ายา` is **total** drug spend, so a patient who stopped
diabetes drugs but stayed on antihypertensives still shows a non-zero figure —
the drug saving here is therefore an *under*-estimate of the diabetes-specific
saving. And this is one hospital's unit costs; it does not generalise to
programme-wide budget impact without the other sites' cost data, which was not
submitted.

---
## 9 · Secondary outcome — determinants of HbA1c improvement

Drug cessation is a clinician's decision as much as a physiological outcome.
Change in HbA1c is a more direct measure of metabolic response, and it is
available for far more patients.

A **linear mixed model with a facility random intercept** is used, since the
outcome is continuous and clustering must again be respected. Baseline HbA1c is
included as a covariate: without it the estimate is dominated by regression to
the mean, and with it the other coefficients are interpretable as "change
beyond what the baseline value alone would predict". That adjustment does not
*remove* regression to the mean, and the caveat in the header still stands.

In [ ]:
ldat = prog.assign(
    female=(prog["sex"] == "Female").astype(float),
    age10=prog["age"] / 10,
    insulin=prog["on_insulin_pre"].fillna(0),
).dropna(subset=["d_hba1c_m6", "hba1c_pre", "age10", "female", "bmi_pre",
                 "n_class_pre", "d_bmi_m6", "facility"])

lmm = smf.mixedlm("d_hba1c_m6 ~ hba1c_pre + age10 + female + bmi_pre + d_bmi_m6 "
                  "+ n_class_pre + insulin",
                  data=ldat, groups=ldat["facility"]).fit()

ci = lmm.conf_int()
lt = pd.DataFrame({"term": lmm.params.index, "beta": lmm.params.values,
                   "lo95": ci[0].values, "hi95": ci[1].values,
                   "p": lmm.pvalues.values})
lt = lt[~lt["term"].isin(["Intercept", "Group Var"])]
lt["term"] = lt["term"].map({
    "hba1c_pre": "Baseline HbA1c, per 1%", "age10": "Age, per 10 years",
    "female": "Female (vs male)", "bmi_pre": "Baseline BMI, per 1 kg/m²",
    "d_bmi_m6": "Change in BMI at 6 mo, per 1 kg/m²",
    "n_class_pre": "Baseline drug classes, per 1",
    "insulin": "On insulin at baseline"}).fillna(lt["term"])
lt.to_csv(TAB / "table10_lmm_hba1c.csv", index=False)

print(f"Outcome: change in HbA1c at 6 months (tiers: {tier_mix(ldat, 'hba1c_m6')})")
print(f"n = {int(lmm.nobs)} patients in {ldat['facility'].nunique()} sites")
print(f"between-site variance = {lmm.cov_re.iloc[0,0]:.4f}, "
      f"residual variance = {lmm.scale:.4f}, "
      f"ICC = {lmm.cov_re.iloc[0,0]/(lmm.cov_re.iloc[0,0]+lmm.scale):.3f}\n")
lt.assign(**{"β (95% CI)": lambda d: d.beta.round(3).astype(str) + " ("
             + d.lo95.round(3).astype(str) + " to " + d.hi95.round(3).astype(str) + ")"},
          p_=lambda d: d.p.map(fmt_p))[["term", "β (95% CI)", "p_"]]

In [ ]:
# does weight loss track glycaemic improvement, at 6 months?
d = prog[["d_bmi_m6", "d_hba1c_m6"]].dropna()
r, pr = stats.pearsonr(d["d_bmi_m6"], d["d_hba1c_m6"])
rho, ps = stats.spearmanr(d["d_bmi_m6"], d["d_hba1c_m6"])

fig, ax = plt.subplots(figsize=(6.4, 4.4))
ax.scatter(d["d_bmi_m6"], d["d_hba1c_m6"], s=15, alpha=0.4, color=dmr_viz.BLUE,
           edgecolors="none")
b = np.polyfit(d["d_bmi_m6"], d["d_hba1c_m6"], 1)
xs = np.linspace(d["d_bmi_m6"].min(), d["d_bmi_m6"].max(), 50)
ax.plot(xs, np.polyval(b, xs), color=dmr_viz.ORANGE, lw=2)
ax.axhline(0, color=dmr_viz.AXIS, lw=1); ax.axvline(0, color=dmr_viz.AXIS, lw=1)
ax.set_xlabel("Change in BMI at 6 months (kg/m²)")
ax.set_ylabel("Change in HbA1c at 6 months (percentage points)")
ax.set_title("Weight change and glycaemic change move together")
ax.annotate(f"n = {len(d)}\nPearson r = {r:.2f} (p {fmt_p(pr)})\n"
            f"Spearman ρ = {rho:.2f} (p {fmt_p(ps)})\n"
            f"slope = {b[0]:.2f} HbA1c points per kg/m²",
            xy=(0.03, 0.97), xycoords="axes fraction", va="top",
            fontsize=8.5, color=dmr_viz.INK_2)
fig.tight_layout()
fig.savefig(FIG / "fig07_bmi_vs_hba1c.png", bbox_inches="tight")
plt.show()

---
## 10 · Programme vs registry — a crude comparison series

The registry is the closest thing this dataset has to a comparison group:
patients with diabetes at a Thai district hospital, **measured at the same
6-month anchor**, who were not enrolled in the DM-Remission programme.

**It is a weak counterfactual and should be read as one.** Different hospital,
different catchment, different period, and a register that includes patients
who died or disengaged. No matching or propensity adjustment is attempted,
because the covariate overlap is poor and pretending otherwise would dress up a
descriptive contrast as a causal one.

What the time anchor does buy is the removal of one specific confounder — the
two cohorts are now compared over the same elapsed time. The formal test below
shows that this substantially shrinks the apparent programme advantage on
HbA1c.

In [ ]:
# Compare on the 6-month anchor, which is exactly what it was built for:
# the registry's own follow-up is much longer, so a last-observed comparison
# would confound the cohort difference with follow-up length.
cmp_rows = []
for name, data in [("Programme cohort", prog), ("Registry (รพ.บ้านตาขุน)", reg)]:
    for a, b, lab, u in pairs_m6:
        r = paired_change(data, a, b, lab, u)
        if r:
            r["cohort"] = name
            r["tiers"] = tier_mix(data.dropna(subset=[a, b]), b)
            cmp_rows.append(r)
cmp = pd.DataFrame(cmp_rows)
cmp.to_csv(TAB / "table11_cohort_comparison.csv", index=False)

cmp_disp = cmp.assign(**{"change (95% CI)": lambda d:
    d.mean_change.round(2).astype(str) + " (" + d.lo95.round(2).astype(str)
    + " to " + d.hi95.round(2).astype(str) + ")  n=" + d.n_pairs.astype(str)}).pivot(
    index="measure", columns="cohort", values="change (95% CI)")
cmp_disp

In [ ]:
fig, ax = plt.subplots(figsize=(8.8, 4.2))
meas = ["HbA1c", "Fasting blood sugar", "BMI", "Waist circumference"]
x = np.arange(len(meas)); w = 0.36
for i_, (name, color) in enumerate([("Programme cohort", dmr_viz.BLUE),
                                    ("Registry (รพ.บ้านตาขุน)", dmr_viz.ORANGE)]):
    sub = cmp[cmp["cohort"] == name].set_index("measure").reindex(meas)
    ax.bar(x + (i_ - 0.5) * w, sub["cohen_dz"], width=w - 0.02, color=color, label=name)
    for xi, (dz, n) in enumerate(zip(sub["cohen_dz"], sub["n_pairs"])):
        if pd.isna(dz):
            continue
        ax.annotate(f"n={int(n)}", xy=(xi + (i_ - 0.5) * w, dz),
                    xytext=(0, -12 if dz < 0 else 5), textcoords="offset points",
                    ha="center", fontsize=8, color=dmr_viz.MUTED)
ax.axhline(0, color=dmr_viz.AXIS, lw=1)
ax.set_xticks(x); ax.set_xticklabels(meas, fontsize=9)
ax.set_ylabel("Cohen's $d_z$\n(negative = improvement)")
ax.legend(loc="upper left")
ax.set_title("Programme cohort vs whole-register extract, both at 6 months\n"
             "Only fasting blood sugar separates them; HbA1c improved in both")
dmr_viz.caption(ax, "Standardised so measures on different scales share one axis. "
                    "Formal between-cohort test below.")
fig.tight_layout()
fig.savefig(FIG / "fig08_cohort_comparison.png", bbox_inches="tight")
plt.show()

#### Is the difference between cohorts real? — a formal test

The bars above are two separate within-person changes. The question the
comparison is actually asking is whether the *change differs between cohorts*,
so that is tested directly (unadjusted, Welch's *t* on the change scores).

The last row of each block repeats the test using the old last-observed
definition, which is where the value of the time anchor becomes visible.

In [ ]:
rows = []
for c, lab, u in [("d_hba1c_m6", "HbA1c", "%"), ("d_fbs_m6", "FBS", "mg/dL"),
                  ("d_bmi_m6", "BMI", "kg/m²"), ("d_waist_m6", "Waist", "cm")]:
    a = pd.to_numeric(prog[c], errors="coerce").dropna()
    b = pd.to_numeric(reg[c], errors="coerce").dropna()
    diff = a.mean() - b.mean()
    se = np.sqrt(a.var(ddof=1) / len(a) + b.var(ddof=1) / len(b))
    rows.append({"measure": lab, "unit": u, "anchor": "6 months",
                 "programme": a.mean(), "n_prog": len(a),
                 "registry": b.mean(), "n_reg": len(b),
                 "difference": diff, "lo95": diff - 1.96 * se, "hi95": diff + 1.96 * se,
                 "p": stats.ttest_ind(a, b, equal_var=False).pvalue})
for c, lab, u in [("d_hba1c", "HbA1c", "%"), ("d_fbs", "FBS", "mg/dL")]:
    a = pd.to_numeric(prog[c], errors="coerce").dropna()
    b = pd.to_numeric(reg[c], errors="coerce").dropna()
    diff = a.mean() - b.mean()
    se = np.sqrt(a.var(ddof=1) / len(a) + b.var(ddof=1) / len(b))
    rows.append({"measure": lab, "unit": u, "anchor": "last observed",
                 "programme": a.mean(), "n_prog": len(a),
                 "registry": b.mean(), "n_reg": len(b),
                 "difference": diff, "lo95": diff - 1.96 * se, "hi95": diff + 1.96 * se,
                 "p": stats.ttest_ind(a, b, equal_var=False).pvalue})

dind = pd.DataFrame(rows)
dind.to_csv(TAB / "table16_between_cohort_difference.csv", index=False)
dind.assign(**{"programme−registry (95% CI)": lambda d:
    d.difference.round(2).astype(str) + " (" + d.lo95.round(2).astype(str)
    + " to " + d.hi95.round(2).astype(str) + ")"},
    p_=lambda d: d.p.map(fmt_p))[
    ["measure", "unit", "anchor", "programme", "registry",
     "programme−registry (95% CI)", "p_"]].round(2)

**This changes the conclusion, and the change is the argument for the time
anchor.**

At a common 6-month timepoint, **only fasting glucose separates the two
cohorts** — a difference of about −28 mg/dL, and not remotely marginal.
HbA1c improved in *both* cohorts and the difference between them
(−0.10 percentage points) is not statistically distinguishable from zero.
BMI and waist show no difference either; the register's BMI fall was, if
anything, slightly larger.

Under the old last-observed definition the apparent HbA1c advantage was
**−0.52 percentage points** — five times larger. That gap was an artefact of
comparing the programme's ~6–12-month change against the register's ~28-month
change, over which the register's glycaemic control drifted upward.

So the earlier reading — "the register did not improve, the programme did" —
**was wrong for HbA1c**, and the corrected statement is narrower:

> At six months, the programme cohort's fasting glucose improved substantially
> more than a non-programme register's. Their HbA1c improvements were similar,
> and their weight changes were similar.

Why the two glycaemic measures diverge is worth asking rather than glossing.
FBS is a same-day measure and is the one most sites recorded with a labelled
6-month timing, whereas the programme cohort's HbA1c timing is `assumed`
throughout (§3) — so some of the gap may be a measurement-timing artefact
rather than physiology. That is a question for the next data collection round,
not something this dataset can settle.

The comparison is now **like-for-like**: both cohorts are measured at the same
6-month timepoint, so the difference is no longer confounded by the registry's
much longer natural follow-up. The formal test below shows what survives that
correction — and it is considerably less than the last-observed comparison
suggested.

Cessation rates are **not** compared here: the registry has no dated medication
data, so it has no 6-month cessation figure (§3), and its last-observed rate is
measured over a median 28 months. Putting the two side by side would compare
different quantities.

---
## 11 · Summary of findings

In [ ]:
print("PROGRAMME COHORT".center(78, "="))
print(f"  {len(prog):,} patients in {prog['facility'].nunique()} sites")
print("  Outcomes anchored at 6 MONTHS after enrolment\n")

print("Effectiveness at 6 months (within-person change, complete pairs)")
for _, r in chg_m6.iterrows():
    print(f"  {r.measure:24s} {r.mean_change:+7.2f} {r.unit:7s} "
          f"(95% CI {r.lo95:+.2f} to {r.hi95:+.2f}), n={int(r.n_pairs):4d}, "
          f"p={fmt_p(r.p_wilcoxon)}   [{r.tiers}]")

print("\n  for comparison, the old last-observed definition:")
for _, r in chg.iterrows():
    if r.measure in set(chg_m6["measure"]):
        print(f"  {r.measure:24s} {r.mean_change:+7.2f} {r.unit:7s} "
              f"(95% CI {r.lo95:+.2f} to {r.hi95:+.2f}), n={int(r.n_pairs):4d}")

print("\nBy provider level (descriptive - see §5.2 before quoting)")
for lv, s in elig_m6.groupby("provider_level"):
    print(f"  cessation @6mo  {lv:20s} {pct_ci(s['drug_stopped_m6'])}")
print(f"  within the CRF layout only, the gap disappears "
      f"(Fisher p = {fmt_p(fisher_p)}) - provider level is confounded")
print(f"  with the data template and cannot be separated from it.")

print("\nOutcome rates at 6 months")
print(f"  Drug cessation @6mo      {pct_ci(elig_m6['drug_stopped_m6'])}")
print(f"  Remission @6mo (upper)   {pct_ci(prog['remission_m6'])}")
print(f"  HbA1c < 7% @6mo          {pct_ci(prog['a1c_lt7_m6'])}")
print(f"  (last-observed cessation {pct_ci(elig['drug_stopped'])})")

print("\nDeterminants of cessation at 6 months (Model A, adjusted, clustered by site)")
for _, r in tA.sort_values("p").iterrows():
    star = "  *" if r.p < 0.05 else "   "
    print(f" {star} {r.term:34s} aOR {r.OR:5.2f} "
          f"({r.lo95:.2f}-{r.hi95:.2f})  p={fmt_p(r.p)}")
print(f"\n  between-site correlation {gA.cov_struct.dep_params:.3f}  |  "
      f"ICC {icc:.3f}  |  C-statistic {c_app-np.mean(opt):.3f} (corrected)")

print("\n" + "REGISTRY COHORT - รพ.บ้านตาขุน".center(78, "="))
print(f"  {len(reg):,} patients, 1 site, median natural follow-up "
      f"{reg['followup_months'].median():.0f} months")
print(f"  {int(reg['active_in_care'].sum())} still in active care; "
      f"{int((reg['member_status'] == 'ตาย').sum())} deceased, "
      f"{int((reg['member_status'] == 'ขาดการรักษาไม่มาติดต่อ').sum())} lost to follow-up")
print("\nAt the same 6-month anchor (comparison series, not programme results)")
for _, r in cmp[cmp["cohort"].str.startswith("Registry")].iterrows():
    print(f"  {r.measure:24s} {r.mean_change:+7.2f} {r.unit:7s} "
          f"(95% CI {r.lo95:+.2f} to {r.hi95:+.2f}), n={int(r.n_pairs):4d}")
print(f"\n  Drug cessation @6mo      {pct_ci(reg['drug_stopped_m6'])}"
      "  <- no dated medication data")
print(f"  Drug cessation, last obs {pct_ci(elig_reg['drug_stopped'])} "
      "over a median 28 months")
print(f"  Drug cost per visit, stoppers: "
      f"{by_grp.loc['Stopped all drugs','drug_pre']:.0f} -> "
      f"{by_grp.loc['Stopped all drugs','drug_post']:.0f} THB; "
      f"lab cost {by_grp.loc['Stopped all drugs','lab_pre']:.0f} -> "
      f"{by_grp.loc['Stopped all drugs','lab_post']:.0f} THB")
print("=" * 78)

### How to read this

**Outcomes are anchored at 6 months.** That is the main methodological change,
and it matters most for the comparison against the registry: the two cohorts
are measured over the same elapsed time. It also shrinks the estimated HbA1c
improvement relative to the old last-observed definition, which tells you the
old figure was partly picking up longer follow-up in the sites that recorded
it.

**The timing tiers are part of the result, not a caveat bolted on.** HbA1c in
the programme cohort is `assumed` almost throughout — the CRF never labels its
HbA1c follow-up against enrolment. Anyone quoting the 6-month HbA1c change
should say the timing is inferred from the template, not recorded.

**The programme's clearest advantage is on fasting glucose, not HbA1c.** At a
common 6-month anchor the programme cohort's fasting glucose improved about
28 mg/dL more than the non-programme register's (p < 0.001), but their HbA1c
improvements were similar (difference −0.10 points, p = 0.17) and their weight
changes were similar too. The old last-observed comparison put the HbA1c gap at
−0.52 points; almost all of that was an artefact of comparing a ~6–12-month
change against a ~28-month one. **Do not quote the earlier figure.**

**This is still not a causal estimate.** The comparison rules out "this is just
what happens to any diabetes cohort measured twice over six months" as an
explanation for the fasting-glucose result. It does not rule out regression to
the mean, selection of motivated patients, or the fact that the two cohorts are
different populations at different hospitals.

**The two key determinants replicate.** Baseline drug burden and baseline HbA1c
predict cessation in both settings (§7.9) — though that comparison runs on the
last-observed outcome, because the registry has no dated medication.

**Site matters as much as patient.** Within the programme, the between-site
correlation and ICC say that which site a patient attends predicts cessation
independently of anything about the patient, and per-site rates span a wide
range. The most actionable finding may be organisational rather than clinical.

**But the provider-level split cannot be attributed to provider level.**
Cessation looks twice as high in primary care (19.2 % vs 9.4 %), and that is a
real difference in the data — but provider level and data template are nearly
the same variable here, and the two templates ask about different points in the
patient's course. Restricting to one template narrows the gap without closing
it (9.4 % vs 17.1 %, Fisher *p* = 0.167), and with 41 patients and 7 events on
the primary-care side that test is **inconclusive, not exculpatory**. §5.2 is
the section to read before anyone writes "primary care deprescribes better" —
the data cannot support that claim, and can no longer refute it either.

**Deprescribing moves cost rather than simply saving it.** Pharmacy spending
falls for patients who stop; laboratory spending rises. A budget case has to
carry both sides.

**What would strengthen the next round of data collection**

1. **Record a date beside every value.** This is the single highest-value
   change. It is the only reason บ้านตาขุน supports a `measured` timing tier
   while every other workbook does not, and it is what forces HbA1c onto the
   `assumed` tier for the whole programme cohort.
2. **Use one template across provider levels.** As things stand, provider level
   and data template are the same variable, so the most interesting service-
   delivery question in the dataset cannot be answered. A single template would
   settle it at a stroke.
3. **Record the date drugs were stopped**, so remission can be assessed against
   the ADA/EASD definition instead of reported as an upper bound — and so
   cessation can be placed on the 6-month clock in the registry too.
4. **Label follow-up columns against enrolment**, not against programme end.
   The CRF's `หลังจบโครงการ 6 เดือน` is a different clock and cannot be pooled
   with `ระหว่างเข้าร่วมโครงการ 6 เดือน`.
5. **Ask the other sites for cost data.** One hospital's costs cannot support a
   programme-wide budget-impact estimate.
6. **Recruit the comparison group deliberately.** A contemporaneous cohort of
   *eligible non-participants from the same sites* would be a real control.

### Outputs written

In [ ]:
print("tables:")
for f in sorted(TAB.glob("*.csv")):
    print(f"  outputs/tables/{f.name}")
print("\nfigures:")
for f in sorted(FIG.glob("*.png")):
    print(f"  outputs/figures/{f.name}")